## RUN THIS ONE FIRST OR IN PARALLEL WITH 12b: vinyl alcohol only. **No GPU needed** (a normal CPU runtime is fine). Just press Run all. About 1.5 h, almost all of it waiting for Gemini (16 agent runs). Results go to Drive folder `fruitful-pes-start-design-vinyl`; send back `nb12_results.zip`. The second notebook (12b) does formamide and writes to a different folder, so the two can run at the same time. Ignore the parts of the text below that mention the other molecule.

---

# NB12a (vinyl alcohol) · Does the agent still win when the search starts well? Three kinds of start, two molecules (Paper B2)
**Fruitful PES Explorer, notebook 12.**

**The question in one line.** In NB10 (vinyl alcohol) the agent got the barrier right from 8 of 8 starts and the best classical rule from 1 of 8. Is that because the agent searches better, or because every search was handed a poor starting point that the agent could recover from and the fixed rules could not?

**What a CPU pilot on the stored NB10 data already showed (no GPU, no Gemini; these numbers are real).** Every search begins with four Skala calls. Two of them were fixed "anchors" at the edges of the map (θ = 0 and θ = 180). That choice was inherited from HCN, where the edges *are* the two wells. On vinyl alcohol the wells sit at θ = 30 and 150, and the edge anchor is 8.1 kcal/mol above the floor of the well that the barrier is measured from. Changing only those two starting points:

| vinyl alcohol, within 1 kcal/mol when the search stops (of 8 starts) | wells from the scout in the start | four random points | edge anchors (NB10) |
|---|---|---|---|
| best classical rule (saddle exploitation + Δ-learning) | **7** | 0 | 1 |
| frozen twin (the prompt's decision order, followed literally) | 6 | 2 | 1 |
| reordered twin (the same order with "sample the well" moved up) | 7 | 6 | 5 |
| anchor-first control | 4 | 2 | 5 |
| LLM agent | **not run yet** | **not run yet** | 8 (NB10) |

So a fixed rule is fine *if someone already put the right well into its starting points*, and it fails if nobody did. The agent has been tested only in the last column.

**The claim this notebook can support (the positive result to aim for).** *Fixed rules need the reference well in their starting design. The agent does not: it reaches the barrier from any of the three starts, and it costs nothing when the start is already good.* That is a claim about **robustness**, it is true to the mechanism we found (the agent moves "sample the well" ahead of "refine the path"), and it does not depend on one lucky choice of starting points. It needs the two empty cells above, and the same table on a second molecule.

**The second molecule: formamide ⇌ formimidic acid** (H₂N–CH=O ⇌ HN=CH–OH). The hydrogen jumps from nitrogen to oxygen. This is the amide–iminol switch of the peptide bond and of the DNA bases, six atoms, same kind of 1,3 hydrogen shift as vinyl alcohol, so the same recipe grid works. The formamide molecule is more stable, so the barrier is measured from the amide well (θ near 20).

## The design
| | |
|---|---|
| molecules | vinyl alcohol → acetaldehyde (Skala grid from NB10, embedded here, nothing to recompute) and formamide → formimidic acid (new Skala grid, Part A, GPU) |
| three kinds of start (4 points each, 8 starts each, same random numbers) | **scout**: the two wells found by the free xTB scout + 2 random points (an *informed* start). **random**: 4 random points (*uninformed*). **edge**: θ = 0 and θ = 180 at the wells' distance + 2 random points (the NB10 start; *misinformed* when the edge is not the well) |
| arms | LLM agent (menu, Gemini, barrier goal) · the ten classical rules of NB2 · frozen twin · reordered twin · anchor-first control |
| scoring | unchanged from NB10: error against the Skala barrier on the same grid, the same stopping rule for every fixed arm, agents stop themselves, paired tests over the 8 starts |
| new measurement | the Skala call at which each search first measured the reference well (a point within 0.5 kcal/mol of its floor): the mechanism, counted |

The agent's prompt is byte-identical to NB10 for vinyl alcohol (checked by a hash in cell 3b). Note what each side knows: the prompt names where the scout puts the wells; the fixed rules see the same scout surface through Δ-learning but have no step that sends them to the well first. The *scout* start hands them that information as two of their four starting points.

## Predictions, written before any agent run in the empty cells
| | prediction (for 8 starts) | status before this notebook |
|---|---|---|
| **P1** | With the scout wells in the start, the best classical rule is within 1 kcal/mol in at least 6 of 8. | known on vinyl (7 of 8); open on formamide |
| **P2** | Without the reference well in the start (random; and edge when the edge anchor is more than 1 kcal/mol above the floor), the best classical rule and the frozen twin are each within 1 kcal/mol in at most 3 of 8. | known on vinyl (0, 1 and 2, 1); open on formamide |
| **P3** | The agent is within 1 kcal/mol in at least 7 of 8 under **every** start. | known for vinyl/edge only (8 of 8); five cells open |
| **P4** | Under random and edge starts the agent measures the reference well before it stops in at least 7 of 8, and earlier than its frozen twin. | open |
| **P5** | Robustness costs nothing: with the scout start the agent's error is not worse than the best classical rule (medians within 0.5 kcal/mol) and it uses no more Skala calls. | open |

The "best classical rule" is fixed in advance as saddle exploitation + Δ-learning (the best rule on HCN, CH₃NC and vinyl alcohol); the best of all ten, picked afterwards, is reported next to it. The reordered twin and the control are reported without a prediction: they show how much of the agent's gain a one-line change of the fixed order would already buy (pilot on vinyl: 5 to 7 of 8).

**How the second molecule was chosen (disclose this in the paper).** The molecule, the direction (θ = 0 on the amide side) and the wall cut-off were fixed after a CPU pilot on the xTB surface and on two cheap stand-in surfaces (PBE and B3LYP). On the stand-ins the fixed rules showed the same pattern as on vinyl (good with the scout wells, 1 to 2 of 8 without). No Skala energy of formamide and no agent run was used for the choice.

## How to run
- **Cheapest first look (no GPU, about 40 min with Gemini):** set `RUN_FORMAMIDE = False` in cell 4. Only the two empty vinyl cells are run (16 agent runs). This alone tells you whether P3 and P5 hold on vinyl.
- **Full run.** Part A needs a GPU (L4 or T4): xTB grid (seconds) and 247 Skala points for formamide, about 20–25 min on an L4, resumable. Part B is CPU: fixed arms about 10–20 min, then 40 agent runs (16 vinyl + 24 formamide) at about 4 min each, run `PARALLEL` at a time (default 2, about 1.5 h). Everything is saved after each run; rerun a cell to resume. If you change to a CPU runtime after Part A, rerun cells 1–6, skip 7–8, continue at 9.
- **Dry run:** `SKALA_STAND_IN = True` (PBE/STO-3G instead of Skala) and `LLM_PROVIDER = "mock"` (the frozen twin plays the agent) test the whole notebook in a few minutes. Their files carry `_STANDIN` / `_MOCK` in the name. **Never report numbers from a dry run.**
- Results go to Drive folder `fruitful-pes-start-design` and into `nb12_results.zip`.

**What changed in the code relative to NB10.** The shared `fpes.py` is untouched. Cell 3b makes two re-labelled copies (text and constants only, every rewrite asserted). Cell 10 swaps the function that draws the four starting points, adds the reordered twin (one extra rule) and the control. Not changed: menu, tags, decision order, frozen twin, surrogate, budget 24, stopping rule (0.5 / 5 / 10), the 1 kcal/mol criterion. Dropped here: the knob agent and the valley goal (they are not part of this question).


In [ ]:
#@title 1. Install (3-6 min). Runtime -> Change runtime type -> GPU first.
import subprocess, sys, importlib
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    print(("ok   " if r.returncode == 0 else "FAIL ") + " ".join(a));
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0

pip("tblite", "scikit-learn", "anthropic", "google-genai", "scipy", "matplotlib")
# Skala: GPU wheel for CUDA 12 (Colab default), pinned to the version verified on Colab (2026.4 gave wrong forces in the
# optimizer notebook). Falls back to the CPU package if the GPU wheel is unavailable. Restart the session after changing it.
SKALA_VERSION = "2026.9"
if not pip(f"skala-cuda12x=={SKALA_VERSION}"):
    pip(f"skala=={SKALA_VERSION}") or pip("skala")
# gpu4pyscf is what makes PBE/B3LYP fast on the GPU; harmless if the Skala wheel already pulled it.
pip("gpu4pyscf-cuda12x", "cupy-cuda12x")

for m in ["pyscf", "skala", "tblite", "sklearn", "anthropic"]:
    try: importlib.import_module(m); print("import ok:", m)
    except Exception as e: print("import FAILED:", m, repr(e)[:120])
try:
    import gpu4pyscf; print("gpu4pyscf available: GPU DFT on")
except Exception as e:
    print("gpu4pyscf NOT available, DFT will run on CPU:", repr(e)[:100])


In [ ]:
#@title 2. Workspace on Google Drive
import os, json, time, numpy as np
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/fruitful-pes-start-design-vinyl'
except Exception:
    WORKDIR = os.path.abspath('./fruitful-pes-start-design-vinyl')
os.makedirs(WORKDIR, exist_ok=True); os.makedirs(f"{WORKDIR}/figures", exist_ok=True)
print("WORKDIR:", WORKDIR)


In [ ]:
#@title 3. Shared code (fpes.py). Identical in all notebooks.
%%writefile fpes.py
"""fpes — shared code for the "Fruitful PES Explorer" notebooks.

HCN <-> HNC on a rigid two-coordinate grid (theta, r) around the C-N centre of mass,
several electronic-structure fidelities (xTB, MACE, PBE, B3LYP, Skala, CCSD(T)),
a grid "oracle" so that acquisition experiments are fast and reproducible,
baseline acquisition policies, and two LLM "Judge" designs (step agent, rule agent).

Energies are stored in Hartree; barriers are reported in kcal/mol.
"""
from __future__ import annotations
import os, json, time, math, dataclasses, textwrap, warnings
import numpy as np
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
try:
    from sklearn.exceptions import ConvergenceWarning
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
except Exception:
    pass

HARTREE_KCAL = 627.5094740631
EV_HARTREE = 1.0 / 27.211386245988
BOHR = 0.529177210903

# ----------------------------------------------------------------------------- geometry
M_C, M_N = 12.0, 14.003074
R_CN_DEFAULT = 1.16  # Angstrom, between HCN (1.153) and HNC (1.169)


def hcn_geometry(theta_deg: float, r: float, R_CN: float = R_CN_DEFAULT):
    """Return [(symbol, (x,y,z)), ...] in Angstrom.

    C and N lie on the z axis with their centre of mass at the origin.
    theta = 0 puts H on the C side (HCN), theta = 180 on the N side (HNC).
    r is the H distance from the C-N centre of mass.
    """
    zc = -R_CN * M_N / (M_C + M_N)
    zn = R_CN * M_C / (M_C + M_N)
    t = math.radians(theta_deg)
    h = (r * math.sin(t), 0.0, -r * math.cos(t))
    return [("C", (0.0, 0.0, zc)), ("N", (0.0, 0.0, zn)), ("H", h)]


def atoms_to_pyscf(atoms):
    return "; ".join(f"{s} {x:.6f} {y:.6f} {z:.6f}" for s, (x, y, z) in atoms)


def make_grid(n_theta: int, n_r: int, theta_range=(0.0, 180.0), r_range=(1.05, 2.30)):
    theta = np.linspace(*theta_range, n_theta)
    r = np.linspace(*r_range, n_r)
    return theta, r


# ----------------------------------------------------------------------------- hardware plan
def detect_gpu():
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=10).stdout.strip()
        return out.split("\n")[0] if out else None
    except Exception:
        return None


def plan_for_hardware(gpu_name: str | None):
    """Grid density and basis per accelerator so that each notebook fits ~3 h."""
    g = (gpu_name or "").upper()
    if "A100" in g or "H100" in g:
        return dict(tier="A100", n_theta=37, n_r=25, basis="def2-tzvp", skala_every=1)
    if "L4" in g:
        return dict(tier="L4", n_theta=31, n_r=21, basis="def2-svp", skala_every=1)
    if "T4" in g:
        return dict(tier="T4", n_theta=25, n_r=17, basis="def2-svp", skala_every=1)
    return dict(tier="CPU", n_theta=13, n_r=9, basis="def2-svp", skala_every=2)


# ----------------------------------------------------------------------------- calculators
class Calc:
    """Uniform interface: calc(atoms) -> dict(E=hartree, converged=bool, niter=int, seconds=float)."""
    name = "base"

    def __call__(self, atoms):
        raise NotImplementedError


class XTB(Calc):
    name = "xtb"

    def __init__(self, method="GFN2-xTB"):
        from tblite.interface import Calculator  # noqa
        self._Calculator = Calculator
        self.method = method
        self._Z = {"H": 1, "C": 6, "N": 7, "O": 8}

    def __call__(self, atoms):
        t = time.time()
        numbers = np.array([self._Z[s] for s, _ in atoms])
        pos = np.array([c for _, c in atoms]) / BOHR
        calc = self._Calculator(self.method, numbers, pos)
        calc.set("verbosity", 0)
        try:
            res = calc.singlepoint()
            e = float(res.get("energy"))
            ok = True
        except Exception:
            e, ok = float("nan"), False
        return dict(E=e, converged=ok, niter=-1, seconds=time.time() - t)


class MACE(Calc):
    name = "mace"

    def __init__(self, model="small", device="cpu"):
        from mace.calculators import mace_off  # noqa
        from ase import Atoms  # noqa
        self._Atoms = Atoms
        self.calc = mace_off(model=model, device=device, default_dtype="float64")

    def __call__(self, atoms):
        t = time.time()
        a = self._Atoms([s for s, _ in atoms], positions=[c for _, c in atoms])
        a.calc = self.calc
        e = a.get_potential_energy() * EV_HARTREE
        return dict(E=float(e), converged=True, niter=-1, seconds=time.time() - t)


class DFT(Calc):
    """PySCF Kohn-Sham. xc in {'pbe','b3lyp',...}. gpu=True uses gpu4pyscf when importable."""

    def __init__(self, xc="pbe", basis="def2-svp", gpu=False, grids_level=3):
        from pyscf import gto, dft  # noqa
        self.gto, self.dft = gto, dft
        self.xc, self.basis, self.gpu, self.grids_level = xc, basis, gpu, grids_level
        self.name = xc
        self._dm0 = None
        if gpu:
            try:
                import gpu4pyscf  # noqa
                self._gpu_ok = True
            except Exception:
                self._gpu_ok = False
        else:
            self._gpu_ok = False

    def reset_guess(self):
        self._dm0 = None

    def __call__(self, atoms):
        t = time.time()
        mol = self.gto.M(atom=atoms_to_pyscf(atoms), basis=self.basis, unit="Angstrom", cart=False, verbose=0)
        mf = self.dft.RKS(mol)
        mf.xc = self.xc
        mf.grids.level = self.grids_level
        if self._gpu_ok:
            mf = mf.to_gpu()
        mf.conv_tol = 1e-9
        mf.max_cycle = 100
        try:
            e = mf.kernel(dm0=self._dm0)
            ok = bool(mf.converged)
            dm = mf.make_rdm1()
            self._dm0 = dm.get() if hasattr(dm, "get") else dm
            niter = int(getattr(mf, "cycles", -1) or -1)
        except Exception:
            e, ok, niter = float("nan"), False, -1
            self._dm0 = None
        return dict(E=float(e), converged=ok, niter=niter, seconds=time.time() - t)


class Skala(Calc):
    """Microsoft Skala learned XC through its PySCF bindings.

    CPU: from skala.pyscf import SkalaKS ; GPU: from skala.gpu4pyscf import SkalaKS.
    Weights download from Hugging Face on first use.
    """
    name = "skala"

    def __init__(self, basis="def2-svp", gpu=False, xc="skala-1.1", with_dftd3=False):
        from pyscf import gto  # noqa
        self.gto = gto
        if gpu:
            from skala.gpu4pyscf import SkalaKS  # noqa
        else:
            from skala.pyscf import SkalaKS  # noqa
        self.SkalaKS, self.basis, self.xc, self.with_dftd3 = SkalaKS, basis, xc, with_dftd3
        self._dm0 = None

    def reset_guess(self):
        self._dm0 = None

    def __call__(self, atoms):
        t = time.time()
        mol = self.gto.M(atom=atoms_to_pyscf(atoms), basis=self.basis, unit="Angstrom", cart=False, verbose=0)
        try:
            try:
                mf = self.SkalaKS(mol, xc=self.xc, with_dftd3=self.with_dftd3)
            except TypeError:  # older signature without with_dftd3
                mf = self.SkalaKS(mol, xc=self.xc)
            mf.conv_tol = 1e-9
            mf.max_cycle = 100
            e = mf.kernel(dm0=self._dm0)
            ok = bool(mf.converged)
            dm = mf.make_rdm1()
            self._dm0 = dm.get() if hasattr(dm, "get") else dm
            niter = int(getattr(mf, "cycles", -1) or -1)
        except Exception as ex:  # keep going; record the failure
            e, ok, niter = float("nan"), False, -1
            self._dm0 = None
            print("Skala failed:", repr(ex)[:200])
        return dict(E=float(e), converged=ok, niter=niter, seconds=time.time() - t)


class CCSDT(Calc):
    """CCSD(T) reference with PySCF (RHF-based). Closed-shell only."""
    name = "ccsd_t"

    def __init__(self, basis="def2-tzvp"):
        from pyscf import gto, scf, cc  # noqa
        self.gto, self.scf, self.cc, self.basis = gto, scf, cc, basis

    def __call__(self, atoms):
        t = time.time()
        mol = self.gto.M(atom=atoms_to_pyscf(atoms), basis=self.basis, unit="Angstrom", cart=False, verbose=0)
        try:
            mf = self.scf.RHF(mol).run(conv_tol=1e-10)
            mycc = self.cc.CCSD(mf).run()
            et = mycc.ccsd_t()
            e = mf.e_tot + mycc.e_corr + et
            ok = bool(mf.converged and mycc.converged)
        except Exception:
            e, ok = float("nan"), False
        return dict(E=float(e), converged=ok, niter=-1, seconds=time.time() - t)


# ----------------------------------------------------------------------------- surfaces with checkpoints
def compute_surface(calc: Calc, theta, r, path: str, every: int = 1, log_every: int = 10, max_minutes: float | None = None):
    """Evaluate calc on the (theta, r) grid; resume from an .npz checkpoint if present.

    `every` > 1 evaluates a coarser sub-grid (every k-th theta and r) and leaves the rest NaN.
    `max_minutes` stops gracefully (checkpoint kept) so a Colab session limit never loses work; rerun to resume.
    Returns dict with E (n_theta, n_r), converged, niter, seconds arrays.
    """
    nt, nr = len(theta), len(r)
    if os.path.exists(path):
        d = dict(np.load(path, allow_pickle=True))
        if d["E"].shape == (nt, nr):
            E, conv, nit, sec = d["E"], d["converged"], d["niter"], d["seconds"]
        else:
            E = np.full((nt, nr), np.nan); conv = np.zeros((nt, nr), bool); nit = -np.ones((nt, nr), int); sec = np.zeros((nt, nr))
    else:
        E = np.full((nt, nr), np.nan); conv = np.zeros((nt, nr), bool); nit = -np.ones((nt, nr), int); sec = np.zeros((nt, nr))
    done_mask = ~np.isnan(E)
    todo = [(i, j) for i in range(0, nt, every) for j in range(0, nr, every) if not done_mask[i, j]]
    print(f"[{calc.name}] grid {nt}x{nr}, every={every}: {len(todo)} points to compute, {done_mask.sum()} cached")
    t0 = time.time()
    last_i = None
    for k, (i, j) in enumerate(todo):
        if hasattr(calc, "reset_guess") and last_i != i:
            calc.reset_guess()  # new theta row: fresh SCF guess
        res = calc(hcn_geometry(theta[i], r[j]))
        E[i, j], conv[i, j], nit[i, j], sec[i, j] = res["E"], res["converged"], res["niter"], res["seconds"]
        last_i = i
        if max_minutes is not None and (time.time() - t0) / 60 > max_minutes:
            np.savez(path, E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)
            print(f"  time box of {max_minutes} min reached after {k+1} points; checkpoint saved, rerun this cell to resume")
            break
        if (k + 1) % log_every == 0 or k == len(todo) - 1:
            np.savez(path, E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)
            el = time.time() - t0
            print(f"  {k+1}/{len(todo)}  theta={theta[i]:.0f} r={r[j]:.2f}  E={E[i,j]:.6f}  "
                  f"conv={conv[i,j]}  {sec[i,j]:.1f}s/pt  elapsed {el/60:.1f} min  ETA {(el/(k+1))*(len(todo)-k-1)/60:.1f} min")
    np.savez(path, E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)
    return dict(E=E, converged=conv, niter=nit, seconds=sec, theta=theta, r=r)


def fill_nan_grid(E, theta, r):
    """Fill NaN entries (coarse Skala sub-grid or failed points) by bilinear interpolation."""
    from scipy.interpolate import griddata
    T, R = np.meshgrid(theta, r, indexing="ij")
    ok = ~np.isnan(E)
    if ok.all():
        return E.copy()
    pts = np.column_stack([T[ok] / 180.0, (R[ok] - r.min()) / (r.max() - r.min())])
    q = np.column_stack([T.ravel() / 180.0, (R.ravel() - r.min()) / (r.max() - r.min())])
    lin = griddata(pts, E[ok], q, method="linear")
    near = griddata(pts, E[ok], q, method="nearest")
    out = np.where(np.isnan(lin), near, lin).reshape(E.shape)
    return out


# ----------------------------------------------------------------------------- barrier on a grid
def basin_minima(E, theta):
    """Indices of the HCN (theta<45) and HNC (theta>135) minima."""
    hcn = np.where(theta < 45.0)[0]
    hnc = np.where(theta > 135.0)[0]
    sub = np.where(np.isnan(E), np.inf, E)
    i1 = np.unravel_index(np.argmin(sub[hcn]), sub[hcn].shape); a = (hcn[i1[0]], i1[1])
    i2 = np.unravel_index(np.argmin(sub[hnc]), sub[hnc].shape); b = (hnc[i2[0]], i2[1])
    return a, b


def bottleneck_barrier(E, theta):
    """Minimum-bottleneck path between the two basins on a 4-connected grid.

    Returns dict(barrier_kcal, saddle=(i,j), E_hcn, E_hnc, isomerization_kcal, threshold).
    """
    nt, nr = E.shape
    Ew = np.where(np.isnan(E), np.inf, E)
    a, b = basin_minima(E, theta)
    order = np.argsort(Ew, axis=None)
    parent = np.arange(nt * nr)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    active = np.zeros(nt * nr, bool)
    ia, ib = a[0] * nr + a[1], b[0] * nr + b[1]
    saddle = None
    for flat in order:
        if not np.isfinite(Ew.flat[flat]):
            break
        active[flat] = True
        i, j = divmod(int(flat), nr)
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr and active[ni * nr + nj]:
                ra, rb = find(flat), find(ni * nr + nj)
                if ra != rb:
                    parent[ra] = rb
        if find(ia) == find(ib):
            saddle = (i, j)
            break
    if saddle is None:
        return dict(barrier_kcal=float("nan"), saddle=None, E_hcn=E[a], E_hnc=E[b], isomerization_kcal=float("nan"))
    thr = E[saddle]
    return dict(barrier_kcal=float((thr - E[a]) * HARTREE_KCAL), saddle=saddle, hcn=a, hnc=b,
                E_hcn=float(E[a]), E_hnc=float(E[b]), isomerization_kcal=float((E[b] - E[a]) * HARTREE_KCAL),
                threshold=float(thr))


def corridor_mask(E, theta, margin_kcal=6.0):
    """Points within `margin` below the bottleneck threshold that connect the two basins
    (the reaction valley near the saddle). Used by path-aware acquisition rules."""
    info = bottleneck_barrier(E, theta)
    if info["saddle"] is None:
        return np.zeros_like(E, bool), info
    thr = info["threshold"]
    low = E <= thr + 1e-12
    # flood fill from the HCN minimum within the low set
    nt, nr = E.shape
    seen = np.zeros_like(low)
    stack = [info["hcn"]]
    while stack:
        i, j = stack.pop()
        if seen[i, j] or not low[i, j]:
            continue
        seen[i, j] = True
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr:
                stack.append((ni, nj))
    band = (E >= thr - margin_kcal / HARTREE_KCAL)
    return seen & band, info


# ----------------------------------------------------------------------------- oracle + surrogate
class GridOracle:
    """Expensive-method lookup on the dense reference grid. Counts calls."""

    def __init__(self, E_ref, theta, r, name="skala"):
        self.E, self.theta, self.r, self.name = E_ref, theta, r, name
        self.calls = 0
        self.log = []

    def nearest(self, th, rr):
        i = int(np.argmin(np.abs(self.theta - th))); j = int(np.argmin(np.abs(self.r - rr)))
        return i, j

    def __call__(self, i, j):
        self.calls += 1
        self.log.append((int(i), int(j), float(self.E[i, j])))
        return float(self.E[i, j])


def features(theta, r, idx):
    """Normalised (theta, r) features for GP regression."""
    th = theta[[i for i, _ in idx]] / 180.0
    rr = (r[[j for _, j in idx]] - r.min()) / (r.max() - r.min())
    return np.column_stack([th, rr])


def gp_predict(theta, r, idx, y, baseline=None, clip_kcal=150.0):
    """GP on sampled points -> (mean, std) over the full grid.

    If `baseline` (n_theta x n_r cheap surface, Hartree) is given, the GP models y - baseline
    (delta-learning) and the prediction adds baseline back.
    """
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
    X = features(theta, r, idx)
    y = np.asarray(y, float)
    if baseline is not None:
        y = y - np.array([baseline[i, j] for i, j in idx])
    elif clip_kcal is not None:  # hard-wall points carry no information about the barrier; soften them
        y = np.minimum(y, y.min() + clip_kcal / HARTREE_KCAL)
    mu, sd = y.mean(), max(y.std(), 1e-6)
    yn = (y - mu) / sd
    kern = C(1.0, (1e-2, 1e2)) * RBF(length_scale=[0.3, 0.3], length_scale_bounds=(0.03, 3.0)) + WhiteKernel(1e-6, (1e-9, 1e-2))
    gp = GaussianProcessRegressor(kern, normalize_y=False, n_restarts_optimizer=2, random_state=0)
    gp.fit(X, yn)
    T, R = np.meshgrid(theta, r, indexing="ij")
    Xg = np.column_stack([T.ravel() / 180.0, (R.ravel() - r.min()) / (r.max() - r.min())])
    m, s = gp.predict(Xg, return_std=True)
    mean = (m * sd + mu).reshape(T.shape)
    std = (s * sd).reshape(T.shape)
    if baseline is not None:
        mean = mean + baseline
    return mean, std


# ----------------------------------------------------------------------------- acquisition baselines
def initial_design(theta, r, n_init, rng):
    """Always include the two basin guesses (HCN, HNC) plus random points."""
    nt, nr = len(theta), len(r)
    idx = [(0, int(np.argmin(np.abs(r - 1.69)))), (nt - 1, int(np.argmin(np.abs(r - 1.53))))]
    while len(idx) < n_init:
        c = (int(rng.integers(nt)), int(rng.integers(nr)))
        if c not in idx:
            idx.append(c)
    return idx


def choose_next(policy, theta, r, idx, mean, std, xtb=None, rng=None):
    nt, nr = len(theta), len(r)
    sampled = np.zeros((nt, nr), bool)
    for i, j in idx:
        sampled[i, j] = True
    if policy == "random":
        cand = np.argwhere(~sampled)
        return tuple(cand[rng.integers(len(cand))])
    if policy == "uniform":
        # farthest-point in feature space (space-filling)
        T, R = np.meshgrid(theta / 180.0, (r - r.min()) / (r.max() - r.min()), indexing="ij")
        pts = np.column_stack([T.ravel(), R.ravel()])
        S = features(theta, r, idx)
        d = np.min(((pts[:, None, :] - S[None, :, :]) ** 2).sum(-1), axis=1).reshape(nt, nr)
        d[sampled] = -1
        return tuple(np.unravel_index(np.argmax(d), d.shape))
    if policy == "uncertainty":
        a = std.copy(); a[sampled] = -1
        return tuple(np.unravel_index(np.argmax(a), a.shape))
    if policy in ("path_uncertainty", "saddle_exploit"):
        mask, info = corridor_mask(mean, theta)
        a = std * (1.0 + 4.0 * mask)
        if policy == "saddle_exploit" and info["saddle"] is not None and not sampled[info["saddle"]]:
            if rng.random() < 0.5:
                return info["saddle"]
        a[sampled] = -1
        return tuple(np.unravel_index(np.argmax(a), a.shape))
    raise ValueError(policy)


def run_policy(policy, oracle: GridOracle, theta, r, budget, seed=0, n_init=4, baseline=None, ref_barrier=None):
    """Sequential acquisition with a GP surrogate; returns per-call barrier estimate and error."""
    rng = np.random.default_rng(seed)
    oracle.calls = 0; oracle.log = []
    idx = initial_design(theta, r, n_init, rng)
    y = [oracle(i, j) for i, j in idx]
    hist = []
    while True:
        mean, std = gp_predict(theta, r, idx, y, baseline=baseline)
        for i, j in idx:  # trust sampled points exactly
            mean[i, j] = y[idx.index((i, j))]
        est = bottleneck_barrier(mean, theta)["barrier_kcal"]
        hist.append(dict(calls=len(idx), barrier=est, error=(est - ref_barrier) if ref_barrier is not None else None))
        if len(idx) >= budget:
            break
        nxt = choose_next(policy, theta, r, idx, mean, std, rng=rng)
        idx.append(tuple(int(v) for v in nxt)); y.append(oracle(*nxt))
    return hist, idx, y


# ----------------------------------------------------------------------------- LLM Judge
STEP_SYSTEM = """You are the Judge in a multi-fidelity exploration of a chemical potential energy surface.
Goal (stated by the user): {usecase}
The surface is HCN <-> HNC on a rigid grid: theta in degrees (0 = H on carbon, HCN; 180 = H on nitrogen, HNC),
r in Angstrom (H distance from the C-N centre of mass). A cheap xTB surface of the whole grid is available for free.
Each expensive call returns one Skala energy at a grid point and costs budget. You have {budget} expensive calls in total.
Use the tools. Prefer calls that reduce uncertainty about the quantity the goal asks for. Explain each choice in one sentence.
When the budget is exhausted, or when you judge the estimate converged, call finish."""

STEP_TOOLS = [
    dict(name="state_summary", description="Current surrogate summary: estimated barrier, predicted saddle, most uncertain points, corridor points, calls used.",
         input_schema=dict(type="object", properties={}, additionalProperties=False), strict=True),
    dict(name="xtb_cut", description="Free: a 1D cut of the cheap xTB surface. axis='theta' scans theta at fixed r; axis='r' scans r at fixed theta.",
         input_schema=dict(type="object", properties=dict(axis=dict(type="string", enum=["theta", "r"]), value=dict(type="number")),
                           required=["axis", "value"], additionalProperties=False), strict=True),
    dict(name="skala_point", description="Spend one expensive call: Skala energy at the grid point nearest (theta, r).",
         input_schema=dict(type="object", properties=dict(theta=dict(type="number"), r=dict(type="number"), why=dict(type="string")),
                           required=["theta", "r", "why"], additionalProperties=False), strict=True),
    dict(name="finish", description="Stop exploring and report the final barrier estimate.",
         input_schema=dict(type="object", properties=dict(reason=dict(type="string")), required=["reason"], additionalProperties=False), strict=True),
]

RULE_SYSTEM = """You design acquisition rules for multi-fidelity exploration of a chemical potential energy surface.
Goal (stated by the user): {usecase}
Surface: HCN <-> HNC, rigid grid over theta (deg, 0=HCN side, 180=HNC side) and r (Angstrom, H distance from the C-N centre of mass).
Write ONE Python function

    def score(theta, r, mean, std, sampled, xtb, corridor, info):
        ...
        return S  # ndarray shape (len(theta), len(r)); the unsampled argmax is the next expensive call

Arguments: theta (1D), r (1D), mean/std (2D surrogate prediction in Hartree over the grid), sampled (2D bool),
xtb (2D cheap surface, Hartree), corridor (2D bool: points in the predicted reaction valley near the bottleneck),
info (dict with keys hcn, hnc, saddle as (i,j) tuples, barrier_kcal). numpy is available as np. 1 Hartree = 627.5 kcal/mol.
Return only the code in a single ```python block. Be specific to the goal; do not just return std."""


class _MockLLM:
    """Deterministic stand-in used when no API key is present so the notebooks run end to end."""

    def step(self, state, calls_left):
        if calls_left <= 0:
            return dict(tool="finish", input=dict(reason="budget exhausted"))
        if state["saddle"] is not None and not state["saddle_sampled"]:
            th, rr = state["saddle_theta_r"]
            return dict(tool="skala_point", input=dict(theta=th, r=rr, why="confirm predicted saddle"))
        th, rr = state["top_uncertain"][0]
        return dict(tool="skala_point", input=dict(theta=th, r=rr, why="largest uncertainty in corridor"))

    def rule(self):
        return textwrap.dedent("""
        def score(theta, r, mean, std, sampled, xtb, corridor, info):
            S = std * (1.0 + 4.0 * corridor)
            if info.get('saddle') is not None and not sampled[info['saddle']]:
                S[info['saddle']] += 10 * std.max()
            return S
        """)


class JudgeRun:
    """Shared machinery for both agent designs: surrogate state, tool execution, logging."""

    def __init__(self, oracle, theta, r, xtb_grid, budget, ref_barrier=None, baseline=None, seed=0, n_init=4):
        self.oracle, self.theta, self.r, self.xtb, self.budget = oracle, theta, r, xtb_grid, budget
        self.ref, self.baseline = ref_barrier, baseline
        rng = np.random.default_rng(seed)
        oracle.calls = 0; oracle.log = []
        self.idx, self.y, self.failed = [], [], set()
        for (i, j) in initial_design(theta, r, n_init, rng):
            self._eval(i, j)
        tries = 0
        while len(self.idx) < n_init and tries < 6 * n_init:   # replace failed initial points (live mode only)
            tries += 1
            c = (int(rng.integers(len(theta))), int(rng.integers(len(r))))
            if c not in self.idx and c not in self.failed:
                self._eval(*c)
        self.hist, self.decisions = [], []
        self.tokens = dict(input=0, output=0)
        self.seed = seed
        self.notes, self.menu_log, self.policy_log, self.summary, self.model = [], [], [], None, None
        self.refresh()

    def _eval(self, i, j):
        e = self.oracle(i, j)
        if np.isfinite(e):
            self.idx.append((i, j)); self.y.append(e)
            return True
        self.failed.add((i, j))
        return False

    def refresh(self):
        self.mean, self.std = gp_predict(self.theta, self.r, self.idx, self.y, baseline=self.baseline)
        for k, (i, j) in enumerate(self.idx):
            self.mean[i, j] = self.y[k]
        self.corridor, self.info = corridor_mask(self.mean, self.theta)
        est = self.info["barrier_kcal"]
        self.hist.append(dict(calls=len(self.idx), barrier=est, error=(est - self.ref) if self.ref is not None else None))

    def sampled_mask(self):
        m = np.zeros_like(self.mean, bool)
        for i, j in list(self.idx) + list(self.failed):
            m[i, j] = True
        return m

    def state(self, k=6):
        s = self.std.copy(); s[self.sampled_mask()] = -1
        top = [tuple(np.unravel_index(f, s.shape)) for f in np.argsort(s, axis=None)[::-1][:k]]
        sad = self.info.get("saddle")
        return dict(
            calls_used=len(self.idx), calls_left=self.budget - len(self.idx),
            barrier_kcal=round(float(self.info["barrier_kcal"]), 2) if np.isfinite(self.info["barrier_kcal"]) else None,
            isomerization_kcal=round(float(self.info["isomerization_kcal"]), 2) if np.isfinite(self.info["isomerization_kcal"]) else None,
            saddle=sad, saddle_theta_r=(float(self.theta[sad[0]]), float(self.r[sad[1]])) if sad else None,
            saddle_sampled=bool(self.sampled_mask()[sad]) if sad else False,
            saddle_std_kcal=round(float(self.std[sad] * HARTREE_KCAL), 2) if sad else None,
            top_uncertain=[(float(self.theta[i]), float(self.r[j])) for i, j in top],
            top_uncertain_std_kcal=[round(float(self.std[i, j] * HARTREE_KCAL), 2) for i, j in top],
            corridor_points=int(self.corridor.sum()),
            sampled=[(float(self.theta[i]), float(self.r[j]), round(float((self.y[k] - min(self.y)) * HARTREE_KCAL), 1)) for k, (i, j) in enumerate(self.idx)],
        )

    def xtb_cut(self, axis, value):
        if axis == "theta":
            j = int(np.argmin(np.abs(self.r - value)))
            prof = (self.xtb[:, j] - np.nanmin(self.xtb)) * HARTREE_KCAL
            return "theta(deg): E_rel(kcal/mol) at r=%.2f\n" % self.r[j] + ", ".join(f"{t:.0f}:{e:.1f}" for t, e in zip(self.theta, prof))
        i = int(np.argmin(np.abs(self.theta - value)))
        prof = (self.xtb[i, :] - np.nanmin(self.xtb)) * HARTREE_KCAL
        return "r(A): E_rel(kcal/mol) at theta=%.0f\n" % self.theta[i] + ", ".join(f"{rr:.2f}:{e:.1f}" for rr, e in zip(self.r, prof))

    def spend(self, th, rr, why=""):
        i, j = self.oracle.nearest(th, rr)
        if (i, j) in self.idx:
            return f"({self.theta[i]:.0f}, {self.r[j]:.2f}) already sampled; no budget spent. E_rel={((self.y[self.idx.index((i,j))]-min(self.y))*HARTREE_KCAL):.2f} kcal/mol"
        if (i, j) in self.failed:
            return f"({self.theta[i]:.0f}, {self.r[j]:.2f}) failed earlier (SCF); no budget spent. Pick another point."
        if len(self.idx) >= self.budget:
            return "budget exhausted"
        if not self._eval(i, j):
            return f"Skala SCF failed at (theta={self.theta[i]:.0f}, r={self.r[j]:.2f}); the point is excluded and no budget is counted. Calls left {self.budget-len(self.idx)}."
        e = self.y[-1]
        self.decisions.append(dict(theta=float(self.theta[i]), r=float(self.r[j]), why=why))
        self.refresh()
        return f"Skala E_rel at (theta={self.theta[i]:.0f}, r={self.r[j]:.2f}) = {((e-min(self.y))*HARTREE_KCAL):.2f} kcal/mol. New barrier estimate {self.info['barrier_kcal']:.2f} kcal/mol. Calls left {self.budget-len(self.idx)}."


# ----------------------------------------------------------------------------- provider-neutral chat (Claude tool calling or Gemini function calling)
class Chat:
    """Tiny adapter: one conversation with function calling.

    send(text=None, tool_results=None) -> (assistant_text, [ {id, name, input} ... ])
    tool_results is a list of (call_id, name, result_string) answering the previous calls.
    """
    provider = "mock"

    def __init__(self, system: str, tools=None, model: str | None = None):
        self.system, self.tools, self.model = system, tools or [], model
        self.tokens = dict(input=0, output=0)

    def send(self, text=None, tool_results=None):
        raise NotImplementedError


class AnthropicChat(Chat):
    provider = "anthropic"
    DEFAULT = "claude-opus-5-5"

    def __init__(self, system, tools=None, model=None):
        super().__init__(system, tools, model or self.DEFAULT)
        import anthropic
        self.client = anthropic.Anthropic()
        self.messages = []

    def send(self, text=None, tool_results=None):
        if tool_results:
            self.messages.append({"role": "user", "content": [dict(type="tool_result", tool_use_id=cid, content=out) for cid, _, out in tool_results]})
        if text:
            self.messages.append({"role": "user", "content": text})
        kw = dict(model=self.model, max_tokens=4000, system=self.system, messages=self.messages)
        if self.tools:
            kw["tools"] = self.tools
        resp = self.client.messages.create(**kw)
        self.tokens["input"] += resp.usage.input_tokens; self.tokens["output"] += resp.usage.output_tokens
        self.messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason == "refusal":
            return "[refused]", []
        txt = "".join(b.text for b in resp.content if b.type == "text")
        calls = [dict(id=b.id, name=b.name, input=dict(b.input)) for b in resp.content if b.type == "tool_use"]
        return txt, calls


# Gemini model ladder: the same ids as the C10H6 optimizer notebook. MODEL_FAST answers every Judge call;
# MODEL_PRO and MODEL_STRONG are used only if MODEL_FAST is unknown to the API (404) or keeps failing.
# Change them from a notebook with fpes.set_gemini_models(fast, pro, strong).
MODEL_FAST = "gemini-3.8-flash"
MODEL_PRO = "gemini-pro-latest"
MODEL_STRONG = "gemini-4-argon"
GEMINI_KEY_VARS = ("GEMINI_API_KEY", "GEMINI_API_KEY_2")   # a second key (optional) is used when the first is rate limited
GEMINI_UNAVAILABLE = set()   # model ids the API answered 404 for in this session; later chats skip them


def set_gemini_models(fast=None, pro=None, strong=None):
    """Set the Gemini ladder used by GeminiChat (blank or None keeps the current id)."""
    global MODEL_FAST, MODEL_PRO, MODEL_STRONG
    MODEL_FAST, MODEL_PRO, MODEL_STRONG = (fast or MODEL_FAST), (pro or MODEL_PRO), (strong or MODEL_STRONG)
    GeminiChat.DEFAULT = MODEL_FAST
    return [MODEL_FAST, MODEL_PRO, MODEL_STRONG]


def gemini_ladder(first=None):
    """Models to try, in order, without duplicates: the requested one first, then fast -> pro -> strong."""
    out = []
    for m in (first, MODEL_FAST, MODEL_PRO, MODEL_STRONG):
        if m and m not in out and m not in GEMINI_UNAVAILABLE:
            out.append(m)
    return out


def _gemini_error_kind(ex):
    """'model' = this model id is not available (go to the next model); 'transient' = rate limit / overload (wait, retry);
    'fatal' = bad request or bad key (raise)."""
    code = getattr(ex, "code", None) or getattr(ex, "status_code", None)
    msg = (str(getattr(ex, "status", "")) + " " + str(ex)).upper()
    if code == 404 or "NOT_FOUND" in msg or "IS NOT FOUND" in msg or "NOT SUPPORTED FOR GENERATECONTENT" in msg:
        return "model"
    if code in (429, 500, 502, 503, 504) or any(k in msg for k in ("RESOURCE_EXHAUSTED", "UNAVAILABLE", "OVERLOADED", "DEADLINE", "INTERNAL", "TIMEOUT", "TIMED OUT")):
        return "transient"
    return "fatal"


class GeminiChat(Chat):
    """Gemini function calling with a small safety net: key rotation, exponential back-off, model fallback.

    self.model is the model that answered the latest call (it changes only if the ladder had to move on);
    self.models_used counts calls per model so the cost column stays honest.
    """
    provider = "gemini"
    DEFAULT = MODEL_FAST
    MAX_ROUNDS_PER_MODEL = 3      # failed rounds (all keys) on one model before moving to the next model
    BACKOFF_BASE_S, BACKOFF_MAX_S = 5.0, 120.0

    def __init__(self, system, tools=None, model=None):
        super().__init__(system, tools, model or MODEL_FAST)
        from google import genai
        from google.genai import types
        self.types = types
        keys = [os.environ.get(v) for v in GEMINI_KEY_VARS] + [os.environ.get("GOOGLE_API_KEY")]
        keys = [k for i, k in enumerate(keys) if k and k not in keys[:i]]
        self.clients = [genai.Client(api_key=k) for k in keys] or [genai.Client()]
        self.client = self.clients[0]
        self.ladder = gemini_ladder(self.model)
        if not self.ladder:
            raise RuntimeError(f"Gemini: every model id was rejected earlier in this session ({sorted(GEMINI_UNAVAILABLE)}); fix the ids with fpes.set_gemini_models(...)")
        self.model = self.ladder[0]
        self.models_used, self.waited_s, self.retries = {}, 0.0, 0
        self.tokens_by_model = {}
        decls = []
        for t in self.tools:
            params = {k: v for k, v in t["input_schema"].items() if k != "additionalProperties"}
            decls.append(types.FunctionDeclaration(name=t["name"], description=t["description"], parameters=params))
        self.config = types.GenerateContentConfig(
            system_instruction=system,
            tools=[types.Tool(function_declarations=decls)] if decls else None,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True) if decls else None,
        )
        self.contents = []
        self._n = 0

    def _generate(self):
        """One generate_content call with key rotation, back-off and model fallback. Raises only when every model failed."""
        import random
        last = None
        for model in list(self.ladder):
            for rnd in range(self.MAX_ROUNDS_PER_MODEL):
                for client in self.clients:
                    try:
                        resp = client.models.generate_content(model=model, contents=self.contents, config=self.config)
                        if model != self.model:
                            print(f"[gemini] answered by {model} (fallback from {self.model})")
                        self.model = model
                        self.models_used[model] = self.models_used.get(model, 0) + 1
                        return resp
                    except Exception as ex:   # noqa: BLE001  (the SDK raises several error classes)
                        last, kind = ex, _gemini_error_kind(ex)
                        if kind == "fatal":
                            raise
                        if kind == "model":
                            GEMINI_UNAVAILABLE.add(model)
                            print(f"[gemini] model {model} not available ({str(ex)[:80]}); trying the next model")
                            break
                        self.retries += 1
                else:   # every key hit a transient error: wait, then another round on the same model
                    wait = min(self.BACKOFF_MAX_S, self.BACKOFF_BASE_S * 2 ** rnd) * (0.8 + 0.4 * random.random())
                    print(f"[gemini] {model}: rate limited / busy ({str(last)[:60]}); waiting {wait:.0f} s")
                    time.sleep(wait); self.waited_s += wait
                    continue
                break   # model not available: leave the rounds loop, go to the next model
        raise RuntimeError(f"Gemini: no model in {self.ladder} answered; last error: {last!r}")

    def send(self, text=None, tool_results=None):
        T = self.types
        if tool_results:
            self.contents.append(T.Content(role="user", parts=[T.Part.from_function_response(name=name, response={"result": out}) for _, name, out in tool_results]))
        if text:
            self.contents.append(T.Content(role="user", parts=[T.Part.from_text(text=text)]))
        resp = self._generate()
        um = getattr(resp, "usage_metadata", None)
        if um is not None:
            ti, to = int(um.prompt_token_count or 0), int(um.candidates_token_count or 0) + int(getattr(um, "thoughts_token_count", 0) or 0)
            self.tokens["input"] += ti; self.tokens["output"] += to
            tm = self.tokens_by_model.setdefault(self.model, dict(input=0, output=0)); tm["input"] += ti; tm["output"] += to
        cand = resp.candidates[0] if resp.candidates else None
        if cand is None or cand.content is None:
            return "", []
        self.contents.append(cand.content)      # kept whole: Gemini 3 function calling needs its thought signatures back
        txt, calls = "", []
        for part in cand.content.parts or []:
            if getattr(part, "function_call", None):
                self._n += 1
                calls.append(dict(id=f"g{self._n}", name=part.function_call.name, input=dict(part.function_call.args or {})))
            elif getattr(part, "text", None) and not getattr(part, "thought", False):
                txt += part.text
        return txt, calls


def chat_factory(provider: str = "auto", model: str | None = None):
    """Return a callable (system, tools) -> Chat, or None for mock mode.

    provider: 'auto' picks Gemini if GEMINI_API_KEY is set, else Claude if ANTHROPIC_API_KEY is set, else mock.
    Colab secrets named GEMINI_API_KEY / GEMINI_API_KEY_2 / ANTHROPIC_API_KEY are read automatically.
    model: blank = MODEL_FAST for Gemini (with the fast -> pro -> strong fallback), claude-opus-5-5 for Claude.
    """
    for var in (*GEMINI_KEY_VARS, "ANTHROPIC_API_KEY"):
        if not os.environ.get(var):
            try:
                from google.colab import userdata  # noqa
                v = userdata.get(var)
                if v:
                    os.environ[var] = v
            except Exception:
                pass
    if provider == "auto":
        provider = "gemini" if os.environ.get("GEMINI_API_KEY") else ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "mock")
    if provider == "mock":
        print("Judge in MOCK mode (scripted policy). Add GEMINI_API_KEY or ANTHROPIC_API_KEY as a Colab secret to use a real model.")
        return None
    if provider == "gemini":
        if not os.environ.get("GEMINI_API_KEY"):
            raise RuntimeError("GEMINI_API_KEY not set")
        n_keys = sum(bool(os.environ.get(v)) for v in GEMINI_KEY_VARS)
        print(f"Judge on Gemini: {gemini_ladder(model)} (first answers; the others are fallbacks) | API keys: {n_keys}")
        return lambda system, tools=None: GeminiChat(system, tools, model or None)
    if provider == "anthropic":
        if not os.environ.get("ANTHROPIC_API_KEY"):
            raise RuntimeError("ANTHROPIC_API_KEY not set")
        return lambda system, tools=None: AnthropicChat(system, tools, model or None)
    raise ValueError(provider)


# $ per million tokens (input, output); used only for the cost column.
# Models not listed here show cost NaN (unknown), never a fake $0. Add a price with fpes.PRICE["model-id"] = (in, out).
PRICE = {"claude-opus-5-5": (4.0, 20.0), "claude-sonnet-5-5": (2.0, 10.0), "claude-haiku-5-5": (0.10, 0.50),
         "gemini-2.5-flash": (0.30, 2.50), "gemini-2.5-pro": (1.25, 10.0)}
_PRICE_WARNED = set()


def cost_usd(model, tokens):
    if not model or str(model).startswith(("frozen", "mock")):
        return 0.0
    if model not in PRICE:
        if model not in _PRICE_WARNED:
            _PRICE_WARNED.add(model)
            print(f"[cost] no price for {model}: cost shows NaN. Set fpes.PRICE['{model}'] = (usd_per_M_in, usd_per_M_out) to fill it.")
        return float("nan")
    pi, po = PRICE[model]
    return (tokens["input"] * pi + tokens["output"] * po) / 1e6


def run_step_agent(judge: JudgeRun, usecase: str, chat_factory=None, max_turns=80, verbose=True, model=None, client=None):
    """Design A: the LLM chooses every expensive point itself, through function calls.

    chat_factory: from fpes.chat_factory(); None = scripted mock. (`client`/`model` kept for backward compatibility.)
    """
    if chat_factory is None and client is not None:
        chat_factory = lambda system, tools=None: AnthropicChat(system, tools, model)
    mock = chat_factory is None
    system = STEP_SYSTEM.format(usecase=usecase, budget=judge.budget)
    chat = None if mock else chat_factory(system, STEP_TOOLS)
    judge.model = "mock" if mock else chat.model
    pending_text, pending_results = "Begin. Call state_summary first.", None
    for turn in range(max_turns):
        if mock:
            st = judge.state()
            act = _MockLLM().step(st, st["calls_left"])
            calls = [dict(id=f"mock{turn}", name=act["tool"], input=act["input"])]
        else:
            txt, calls = chat.send(text=pending_text, tool_results=pending_results)
            judge.tokens = chat.tokens; judge.model = chat.model
            pending_text, pending_results = None, None
            if txt == "[refused]":
                print("model refused; stopping"); break
            if not calls:
                pending_text = "Continue: call a tool (skala_point, xtb_cut, state_summary) or finish."
                continue
        results, done = [], False
        for c in calls:
            name, inp = c["name"], c["input"]
            if name == "state_summary":
                out = json.dumps(judge.state())
            elif name == "xtb_cut":
                out = judge.xtb_cut(inp["axis"], float(inp["value"]))
            elif name == "skala_point":
                out = judge.spend(float(inp["theta"]), float(inp["r"]), str(inp.get("why", "")))
                if verbose:
                    print(f"  call {len(judge.idx):2d}: ({float(inp['theta']):.0f}, {float(inp['r']):.2f}) {str(inp.get('why',''))[:70]} -> {out[:60]}")
            elif name == "finish":
                out = "finished"; done = True
            else:
                out = f"unknown tool {name}"
            results.append((c["id"], name, out))
        pending_results = results
        if done or len(judge.idx) >= judge.budget:
            break
    return judge


def _compile_rule(code: str):
    code = code.strip()
    if "```" in code:
        code = code.split("```python")[-1].split("```")[0] if "```python" in code else code.split("```")[1]
    ns = {"np": np, "math": math}
    exec(compile(code, "<rule>", "exec"), ns)
    return ns["score"], code


def run_rule_agent(judge: JudgeRun, usecase: str, chat_factory=None, review_every=10, max_rewrites=3, verbose=True, model=None, client=None):
    """Design B: the LLM writes an acquisition rule once; code executes it; the LLM may rewrite after each review."""
    if chat_factory is None and client is not None:
        chat_factory = lambda system, tools=None: AnthropicChat(system, tools, model)
    mock = chat_factory is None
    chat = None if mock else chat_factory(RULE_SYSTEM.format(usecase=usecase), None)
    judge.model = "mock" if mock else chat.model
    rules = []

    def ask(prompt):
        if mock:
            return _MockLLM().rule()
        txt, _ = chat.send(text=prompt)
        judge.tokens = chat.tokens; judge.model = chat.model
        return txt

    score_fn, code = _compile_rule(ask("Write the initial acquisition rule.")); rules.append(code)
    rewrites = 0
    while len(judge.idx) < judge.budget:
        try:
            S = np.asarray(score_fn(judge.theta, judge.r, judge.mean, judge.std, judge.sampled_mask(), judge.xtb, judge.corridor, judge.info), float)
            S = np.where(np.isfinite(S), S, -np.inf)
        except Exception as ex:
            if verbose: print("rule failed, falling back to uncertainty:", repr(ex)[:100])
            S = judge.std.copy()
        S[judge.sampled_mask()] = -np.inf
        i, j = np.unravel_index(int(np.argmax(S)), S.shape)
        judge.spend(float(judge.theta[i]), float(judge.r[j]), why=f"rule v{len(rules)}")
        if verbose:
            print(f"  call {len(judge.idx):2d}: ({judge.theta[i]:.0f}, {judge.r[j]:.2f}) barrier est {judge.info['barrier_kcal']:.2f}")
        if len(judge.idx) % review_every == 0 and rewrites < max_rewrites and len(judge.idx) < judge.budget:
            review = (f"Review after {len(judge.idx)} calls. State: {json.dumps(judge.state(k=4))}. "
                      "Barrier estimates so far (kcal/mol): " + ", ".join(f"{h['barrier']:.1f}" for h in judge.hist[-review_every:] if np.isfinite(h['barrier'])) +
                      ". Keep the rule or return an improved one in a single ```python block.")
            try:
                score_fn, code = _compile_rule(ask(review)); rules.append(code); rewrites += 1
            except Exception as ex:
                if verbose: print("rewrite unusable, keeping previous rule:", repr(ex)[:100])
    judge.rules = rules
    return judge


# ----------------------------------------------------------------------------- Designs M (menu judge) and K (knob tuner)
# Design M: code builds a short, stratified menu of candidate grid points each turn; the LLM picks one by id.
# Design K: the LLM sets the weights of a fixed acquisition formula once per round; code runs the round.
# Both have a "frozen policy" twin (no LLM) that uses the same menus / the same formula, which is the ablation
# that measures the agent's contribution. Both also serve as the mock Judge when no API key is present.

_SHADES = " .:-=+*#%@"
KNOB_NAMES = ("w_unc", "w_path", "w_gap", "w_saddle", "w_spread", "w_wall")


def ascii_map(Z, theta, r, sampled, mark=None, lo=0.0, hi=80.0):
    """Small character map for the LLM: rows = r (top = largest), columns = theta. 'o' sampled, 'S' marked cell."""
    lines = []
    for j in range(len(r) - 1, -1, -1):
        row = ""
        for i in range(len(theta)):
            if mark is not None and (i, j) == tuple(mark):
                row += "S"
            elif sampled[i, j]:
                row += "o"
            else:
                v = (Z[i, j] - lo) / (hi - lo + 1e-12)
                row += _SHADES[min(9, max(0, int(v * 9.999)))] if np.isfinite(v) else "?"
        lines.append(f"r={r[j]:.2f} |{row}|")
    lines.append(" " * 8 + "θ=0" + " " * max(1, len(theta) - 7) + "θ=180")
    return "\n".join(lines)


def _norm01(a):
    a = np.where(np.isfinite(a), a, 0.0)
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo + 1e-12)


def judge_features(J):
    """Per-cell features: predicted E_rel and sigma (kcal/mol), |xTB - surrogate| gap (kcal/mol), grid distance to nearest sample."""
    sig = J.std * HARTREE_KCAL
    mu = (J.mean - J.mean.min()) * HARTREE_KCAL
    gap = np.abs((J.mean - J.mean.min()) - (J.xtb - np.nanmin(J.xtb))) * HARTREE_KCAL
    nt, nr = J.mean.shape
    T, R = np.meshgrid(np.arange(nt), np.arange(nr), indexing="ij")
    pts = np.column_stack([T.ravel(), R.ravel()]).astype(float)
    S = np.array(J.idx, float)
    dist = np.sqrt(((pts[:, None, :] - S[None, :, :]) ** 2).sum(-1)).min(1).reshape(nt, nr)
    return mu, sig, gap, dist


def build_menu(J, rng=None, box=None, want=None, k_max=8):
    """Stratified candidate menu: one unsampled cell per criterion, duplicates removed.

    box = (theta_min, theta_max, r_min, r_max) restricts the search (probe).
    want in {uncertainty, disagreement, low_energy, spread} ranks the whole box by one criterion instead (probe).
    """
    rng = rng if rng is not None else np.random.default_rng(getattr(J, "seed", 0))
    mu, sig, gap, dist = judge_features(J)
    allowed = ~J.sampled_mask()
    if box is not None:
        tmin, tmax, rmin, rmax = [float(x) for x in box]
        allowed &= (J.theta[:, None] >= tmin) & (J.theta[:, None] <= tmax) & (J.r[None, :] >= rmin) & (J.r[None, :] <= rmax)
    cands = []

    def add(tag, score, k=1, positive=False):
        s = np.where(allowed, score, -np.inf)
        for flat in np.argsort(s, axis=None)[::-1][:k]:
            i, j = (int(x) for x in np.unravel_index(int(flat), s.shape))
            if not np.isfinite(s[i, j]) or (positive and s[i, j] <= 0):
                break
            if all((i, j) != c[0] for c in cands):
                cands.append(((i, j), tag))

    if want is not None:
        crit = {"uncertainty": sig, "disagreement": gap, "low_energy": -mu, "spread": dist}.get(want, sig)
        add(str(want), crit, k=k_max)
    else:
        sad = J.info.get("saddle")
        if sad is not None and allowed[sad]:
            cands.append((tuple(int(x) for x in sad), "saddle guess"))
        add("max-σ on path", sig * J.corridor, positive=True)
        add("max-σ anywhere", sig)
        add("xTB≠GP on path", gap * J.corridor, positive=True)
        add("xTB≠GP anywhere", gap)
        add("far from samples", dist)
        basin_added = False
        for key in ("hcn", "hnc"):
            b = J.info.get(key)
            if b is not None and allowed[b] and all(tuple(b) != c[0] for c in cands):
                cands.append((tuple(int(x) for x in b), f"{key.upper()} basin min, unsampled")); basin_added = True; break
        if not basin_added:
            add("low E, unsampled", -mu)
        free = np.argwhere(allowed)
        if len(free):
            i, j = (int(x) for x in free[rng.integers(len(free))])
            if all((i, j) != c[0] for c in cands):
                cands.append(((i, j), "random"))
    rows = []
    for n, ((i, j), tag) in enumerate(cands[:k_max], 1):
        rows.append(dict(id=n, i=i, j=j, theta=float(J.theta[i]), r=float(J.r[j]), tag=tag,
                         E_pred=float(mu[i, j]), sigma=float(sig[i, j]), gap=float(gap[i, j]),
                         on_path=bool(J.corridor[i, j]), dist=float(dist[i, j])))
    return rows


def menu_text(rows):
    out = ["MENU (pick one by id)", " id    θ     r  why it is here              E_pred    σ   gap path dist"]
    for m in rows:
        out.append(f" [{m['id']}] {m['theta']:>3.0f}  {m['r']:.2f}  {m['tag']:<26} {m['E_pred']:>7.1f} {m['sigma']:>4.1f} {m['gap']:>5.1f}  {'Y' if m['on_path'] else 'n'}  {m['dist']:>4.1f}")
    return "\n".join(out)


def state_block(J):
    """What the LLM reads every turn: budget, estimate and trend, saddle and basins, two ASCII maps, its own notes."""
    mu, sig, gap, dist = judge_features(J)
    sampled = J.sampled_mask(); sad = J.info.get("saddle"); hcn, hnc = J.info.get("hcn"), J.info.get("hnc")
    est = J.info["barrier_kcal"]
    traj = ", ".join(f"{h['barrier']:.1f}" for h in J.hist[-6:] if np.isfinite(h["barrier"]))
    out = [f"BUDGET   used {len(J.idx)} / {J.budget}  (left {J.budget - len(J.idx)})",
           f"ESTIMATE barrier {est:.1f} kcal/mol" + (f", σ at saddle {sig[sad]:.1f}" if sad is not None else ""),
           f"         last estimates: {traj}"]
    if sad is not None:
        out.append(f"SADDLE   θ={J.theta[sad[0]]:.0f}°, r={J.r[sad[1]]:.2f} Å  ({'sampled' if sampled[sad] else 'NOT yet sampled at Skala level'})")
    if hcn is not None and hnc is not None:
        out.append(f"BASINS   HCN θ={J.theta[hcn[0]]:.0f} r={J.r[hcn[1]]:.2f} ({'sampled' if sampled[hcn] else 'unsampled'})")
        out.append(f"         HNC θ={J.theta[hnc[0]]:.0f} r={J.r[hnc[1]]:.2f} ({'sampled' if sampled[hnc] else 'unsampled'}); ΔE(HNC−HCN) {J.info['isomerization_kcal']:.1f}")
    if J.decisions:
        d = J.decisions[-1]
        out.append(f"LAST     ({d['theta']:.0f}, {d['r']:.2f}) because: {str(d['why'])[:120]}")
    out += ["", "MAP predicted E_rel, kcal/mol  (' '=0 … '@'≥80)", "    o = sampled, S = saddle guess",
            ascii_map(mu, J.theta, J.r, sampled, sad, 0, 80),
            "", "MAP uncertainty σ, kcal/mol  (' '=0 … '@'≥10)",
            ascii_map(sig, J.theta, J.r, sampled, sad, 0, 10),
            "", "NOTES (yours): " + ("; ".join(J.notes[-6:]) if J.notes else "none yet")]
    return "\n".join(out)


def pick_from_menu(J, menu, cid, why):
    """Spend one expensive call on menu item `cid`; log which kind of candidate was taken."""
    try:
        cid = int(cid)
    except Exception:
        return f"id must be an integer 1..{len(menu)}"
    m = next((x for x in menu if x["id"] == cid), None)
    if m is None:
        return f"no candidate with id {cid}; ids are 1..{len(menu)}"
    before = J.info["barrier_kcal"]
    out = J.spend(m["theta"], m["r"], str(why))
    J.menu_log.append(dict(call=len(J.idx), id=cid, tag=m["tag"], theta=m["theta"], r=m["r"], why=str(why)[:200],
                           offered=[x["tag"] for x in menu]))
    sad = J.info.get("saddle")
    extra = f" Barrier was {before:.1f}."
    if sad is not None:
        extra += f" Saddle guess now θ={J.theta[sad[0]]:.0f}, r={J.r[sad[1]]:.2f}; σ there {J.std[sad] * HARTREE_KCAL:.1f}."
    return out + extra


MENU_ORDER = """
DEFAULT DECISION ORDER (the GOAL overrides it; take the first that applies):
 1. Saddle guess unsampled and sigma there above 0.5: pick it.
 2. An on-path candidate has sigma above 1 or gap above 3: pick the one that most threatens the
    estimate (largest sigma + gap on the path).
 3. A basin minimum is unsampled: pick it; the barrier is measured from it.
 4. Estimate moved less than 0.5 over the last 5 picks and nothing above applies: finish.
 5. Otherwise the most uncertain candidate with E_pred below 80.
"""

MENU_SYSTEM = """You are the Judge of a multi-fidelity exploration of a chemical potential energy surface.

GOAL (from the user, verbatim): {usecase}
METRIC: the HCN -> HNC bottleneck barrier (the highest energy on the lowest connecting path
across the grid), within 1 kcal/mol of the converged answer, using as few expensive calls as
possible. Both count: the error and the number of calls.

THE SURFACE: HCN <-> HNC on a rigid grid. theta (degrees): 0 = H on the carbon side (HCN),
180 = H on the nitrogen side (HNC). r (Angstrom): distance of H from the C-N centre of mass.
Expect two basins (theta near 0-15 and near 165-180), one saddle around theta 70-100 at small r,
and steep walls (E_rel above 80 kcal/mol) at small r inside the basins and at large r everywhere.
The cheap xTB surface is roughly right about shape and often wrong about heights near the saddle.

FREE: the state block, the maps, the menu, probe, xtb_cut, note.  COSTS: pick, one Skala energy.
You have {budget} picks.

EACH TURN you get a state block (budget, estimate and its trend, saddle guess, two ASCII maps:
predicted E_rel and uncertainty sigma; 'o' = sampled, 'S' = saddle guess) and a MENU of
candidates with: E_pred, sigma, gap = |E_rel from xTB - E_rel from the surrogate|,
path = Y if the point lies in the reaction valley near the bottleneck, dist = grid steps to the
nearest sampled point.
{order}
RULES: exactly one function call per turn. Never pick E_pred above 80 unless the GOAL is about
walls. Never pick a point marked already sampled. Start 'why' with what the last result changed,
then the reason for this pick; 30 words at most. Call note() when you form or drop a hypothesis;
notes are shown back to you. A good estimate with budget left beats a spent budget."""

MENU_TOOLS = [
    dict(name="pick", description="Spend one expensive call on a menu candidate. The only function that costs budget.",
         input_schema=dict(type="object", properties=dict(id=dict(type="integer", description="menu id"), why=dict(type="string", description="what the last result changed, then the reason; 30 words max")),
                           required=["id", "why"], additionalProperties=False), strict=True),
    dict(name="probe", description="Free: rebuild the menu inside a box of the grid, ranked by one criterion.",
         input_schema=dict(type="object", properties=dict(theta_min=dict(type="number"), theta_max=dict(type="number"), r_min=dict(type="number"), r_max=dict(type="number"),
                                                          want=dict(type="string", enum=["uncertainty", "disagreement", "low_energy", "spread"])),
                           required=["theta_min", "theta_max", "r_min", "r_max", "want"], additionalProperties=False), strict=True),
    STEP_TOOLS[1],  # xtb_cut
    dict(name="note", description="Free: store a short hypothesis or observation; it is shown back to you in every later state block.",
         input_schema=dict(type="object", properties=dict(text=dict(type="string")), required=["text"], additionalProperties=False), strict=True),
    dict(name="finish", description="Stop exploring and report the final barrier estimate.",
         input_schema=dict(type="object", properties=dict(barrier_kcal=dict(type="number"), confidence=dict(type="string", enum=["high", "medium", "low"]), summary=dict(type="string")),
                           required=["barrier_kcal", "confidence", "summary"], additionalProperties=False), strict=True),
]

KNOB_SYSTEM = """You set the sampling policy of a multi-fidelity exploration of a chemical potential energy surface.

GOAL (from the user, verbatim): {usecase}
METRIC: the HCN -> HNC bottleneck barrier (the highest energy on the lowest connecting path
across the grid), within 1 kcal/mol of the converged answer, using as few expensive calls as
possible. Both count: the error and the number of calls.

THE SURFACE: HCN <-> HNC on a rigid grid. theta (degrees): 0 = H on the carbon side (HCN),
180 = H on the nitrogen side (HNC). r (Angstrom): distance of H from the C-N centre of mass.
Expect two basins (theta near 0-15 and near 165-180), one saddle around theta 70-100 at small r,
and steep walls (E_rel above 80 kcal/mol) at small r inside the basins and at large r everywhere.
The cheap xTB surface is roughly right about shape and often wrong about heights near the saddle.

FREE: the review block and xtb_cut.  COSTS: set_policy runs round_size Skala calls at once.
You have {budget} calls in total. Code picks each point by maximising
  score = w_unc*sigma + w_path*(sigma on the path) + w_gap*|xTB - GP| + w_saddle*[saddle guess]
        + w_spread*(distance to sampled points) - w_wall*[E_pred above 80]
with every term scaled 0..1 over the grid. You choose the six weights (0 to 2) and round_size (4-10).

DEFAULT SCHEDULE (the GOAL overrides it): early rounds locate (w_saddle, w_path high, w_spread
moderate, large rounds); middle rounds hunt disagreement (w_gap up); late rounds confirm (w_path
only, small rounds). Finish when the barrier trajectory is flat within 0.5 over two rounds.

FORMAT: one function call per turn. In 'why', name the number in the review that made you change
each weight you changed. Keeping the same weights is allowed; say why."""

KNOB_TOOLS = [
    dict(name="set_policy", description="Set the acquisition weights and run one round of expensive calls. Returns a review of the round.",
         input_schema=dict(type="object", properties=dict(w_unc=dict(type="number"), w_path=dict(type="number"), w_gap=dict(type="number"), w_saddle=dict(type="number"),
                                                          w_spread=dict(type="number"), w_wall=dict(type="number"), round_size=dict(type="integer"), why=dict(type="string")),
                           required=list(KNOB_NAMES) + ["round_size", "why"], additionalProperties=False), strict=True),
    STEP_TOOLS[1],  # xtb_cut
    MENU_TOOLS[-1],  # finish
]


def acquisition(J, w):
    """Weighted acquisition over the grid (design K); sampled cells are -inf."""
    mu, sig, gap, dist = judge_features(J)
    sad = np.zeros_like(sig)
    if J.info.get("saddle") is not None:
        sad[J.info["saddle"]] = 1.0
    A = (w.get("w_unc", 0.0) * _norm01(sig) + w.get("w_path", 0.0) * _norm01(sig * J.corridor) + w.get("w_gap", 0.0) * _norm01(gap)
         + w.get("w_saddle", 0.0) * sad + w.get("w_spread", 0.0) * _norm01(dist) - w.get("w_wall", 0.0) * (mu > 80))
    A[J.sampled_mask()] = -np.inf
    return A


def _sigma_path_max(J):
    sig = J.std * HARTREE_KCAL
    return float((sig * J.corridor).max()) if J.corridor.any() else float(sig.max())


def run_round(J, w, n, why=""):
    """Run one round of design K with weights w; returns the review block for the LLM."""
    w = {k: float(min(2.0, max(0.0, float(w.get(k, 0.0))))) for k in KNOB_NAMES}
    n = int(min(12, max(1, int(n))))
    before = _sigma_path_max(J)
    rows = []
    label = " ".join(f"{k}={v:g}" for k, v in w.items())
    for _ in range(n):
        if len(J.idx) >= J.budget:
            break
        A = acquisition(J, w)
        if not np.isfinite(A).any():
            break
        i, j = np.unravel_index(int(np.argmax(A)), A.shape)
        J.spend(float(J.theta[i]), float(J.r[j]), why=f"round {len(J.policy_log) + 1}: {label}")
        rows.append((float(J.theta[i]), float(J.r[j]), float((J.y[-1] - min(J.y)) * HARTREE_KCAL), float(J.info["barrier_kcal"])))
    after = _sigma_path_max(J)
    J.policy_log.append(dict(round=len(J.policy_log) + 1, weights=w, round_size=n, why=str(why)[:300], calls=len(J.idx),
                             barrier=float(J.info["barrier_kcal"]), sigma_path_before=before, sigma_path_after=after))
    return review_block(J, w, rows, before, after)


def review_block(J, w, rows, before, after):
    sampled = J.sampled_mask(); sad = J.info.get("saddle"); sig = J.std * HARTREE_KCAL
    out = ["ROUND DONE with weights", "  " + " ".join(f"{k}={v:g}" for k, v in w.items()),
           f"BUDGET used {len(J.idx)} / {J.budget}",
           "POINTS THIS ROUND  (θ, r) -> E_rel ; barrier estimate after"]
    out += [f"  ({th:>3.0f}, {rr:.2f}) -> {e:>6.1f} ; {b:.1f}" for th, rr, e, b in rows]
    out.append("BARRIER trajectory (all rounds):")
    out.append("  " + ", ".join(f"{h['barrier']:.1f}" for h in J.hist[-14:] if np.isfinite(h["barrier"])))
    out.append(f"σ on path: max {before:.1f} -> {after:.1f} kcal/mol")
    if sad is not None:
        out.append(f"saddle guess θ={J.theta[sad[0]]:.0f} r={J.r[sad[1]]:.2f} ({'sampled' if sampled[sad] else 'unsampled'}), σ there {sig[sad]:.1f}")
    out.append("MAP σ (o=sampled, S=saddle guess)")
    out.append(ascii_map(sig, J.theta, J.r, sampled, sad, 0, 10))
    out.append(f"Calls left {J.budget - len(J.idx)}. Reply with set_policy(...) for the next round, or finish(...).")
    return "\n".join(out)


def _flat(J, k=5, tol=0.5):
    b = [h["barrier"] for h in J.hist[-(k + 1):] if np.isfinite(h["barrier"])]
    return len(b) >= k + 1 and (max(b) - min(b)) < tol


def frozen_menu_choice(J, menu, rule="order"):
    """Chooser without an LLM. rule='order' follows MENU_ORDER literally (mock Judge and frozen twin);
    rule='max_sigma' always takes the most uncertain candidate below the wall."""
    def act(m, why):
        return dict(name="pick", input=dict(id=m["id"], why=why))
    below = [m for m in menu if m["E_pred"] < 80] or menu
    if rule == "max_sigma":
        return act(max(below, key=lambda m: m["sigma"]), "frozen: most uncertain candidate")
    sad = next((m for m in menu if m["tag"] == "saddle guess"), None)
    if sad is not None and sad["sigma"] > 0.5:
        return act(sad, "rule 1: saddle guess unsampled")
    path = [m for m in menu if m["on_path"] and (m["sigma"] > 1 or m["gap"] > 3)]
    if path:
        return act(max(path, key=lambda m: m["sigma"] + m["gap"]), "rule 2: on-path candidate threatens the estimate")
    basin = next((m for m in menu if "basin" in m["tag"]), None)
    if basin is not None:
        return act(basin, "rule 3: basin minimum unsampled")
    if _flat(J):
        return dict(name="finish", input=dict(barrier_kcal=float(J.info["barrier_kcal"]), confidence="medium", summary="rule 4: estimate flat over the last 5 picks"))
    return act(max(below, key=lambda m: m["sigma"]), "rule 5: most uncertain candidate below the wall")


def frozen_knob_schedule(J, round_no):
    """Weights without an LLM: the default schedule of KNOB_SYSTEM (mock Judge and frozen twin)."""
    if round_no == 1:
        return dict(w_unc=0.2, w_path=1.0, w_gap=0.3, w_saddle=1.0, w_spread=0.3, w_wall=1.0, round_size=8, why="frozen schedule: locate")
    if round_no == 2:
        return dict(w_unc=0.1, w_path=1.5, w_gap=0.6, w_saddle=0.5, w_spread=0.0, w_wall=1.0, round_size=6, why="frozen schedule: hunt disagreement")
    return dict(w_unc=0.0, w_path=1.0, w_gap=0.2, w_saddle=0.0, w_spread=0.0, w_wall=1.0, round_size=6, why="frozen schedule: confirm")


def run_menu_agent(J, usecase, chat_factory=None, scaffold=True, frozen=None, max_free=3, verbose=True):
    """Design M: the LLM picks one tagged candidate per turn from a code-built menu.

    chat_factory None, or frozen in {'order', 'max_sigma'}: no LLM, the frozen chooser decides (mock mode / frozen twin).
    scaffold False removes the default decision order from the prompt (ablation).
    Each pick is a fresh conversation; memory is the notes and the state block, so cost stays flat.
    """
    rng = np.random.default_rng(getattr(J, "seed", 0))
    system = MENU_SYSTEM.format(usecase=usecase, budget=J.budget, order=MENU_ORDER if scaffold else "\n")
    use_llm = chat_factory is not None and frozen is None
    J.model = None if use_llm else f"frozen-{frozen or 'order'}"
    done = False
    while len(J.idx) < J.budget and not done:
        menu = build_menu(J, rng)
        if not menu:
            break
        chat = None
        if use_llm:
            chat = chat_factory(system, MENU_TOOLS); J.model = chat.model
            txt, calls = chat.send(text=state_block(J) + "\n\n" + menu_text(menu))
        else:
            a = frozen_menu_choice(J, menu, frozen or "order")
            calls = [dict(id="frozen", name=a["name"], input=a["input"])]
        free = 0
        for _guard in range(12):
            if not calls:
                if chat is None:
                    break
                txt, calls = chat.send(text="Call exactly one function: pick, probe, xtb_cut, note or finish.")
                continue
            c = calls[0]; name, inp = c["name"], c["input"]
            if name == "pick":
                out = pick_from_menu(J, menu, inp.get("id", 0), inp.get("why", ""))
                if "no candidate" in out or "must be an integer" in out:
                    if chat is None:
                        break
                    txt, calls = chat.send(tool_results=[(c["id"], name, out)]); continue
                if verbose:
                    print(f"  call {len(J.idx):2d}: {out[:70]} | {str(inp.get('why', ''))[:60]}")
                break
            if name == "finish":
                J.summary = dict(inp); done = True
                if verbose:
                    print(f"  finish after {len(J.idx)} calls: {str(inp.get('summary', ''))[:80]}")
                break
            if name == "probe":
                try:
                    menu = build_menu(J, rng, box=(inp["theta_min"], inp["theta_max"], inp["r_min"], inp["r_max"]), want=inp.get("want"))
                    out = menu_text(menu) if menu else "no unsampled points in that box"
                except Exception as ex:
                    out = f"probe failed: {ex!r}"
            elif name == "xtb_cut":
                out = J.xtb_cut(inp.get("axis", "theta"), float(inp.get("value", 0)))
            elif name == "note":
                J.notes.append(str(inp.get("text", ""))[:200]); out = "noted"
            else:
                out = f"unknown function {name}"
            free += 1
            if free >= max_free:
                out += "\nFree calls for this turn are used up: pick or finish now."
            if chat is None:
                break
            txt, calls = chat.send(tool_results=[(c["id"], name, out)])
        if chat is not None:
            J.tokens["input"] += chat.tokens["input"]; J.tokens["output"] += chat.tokens["output"]; J.model = chat.model
    return J


def run_knob_agent(J, usecase, chat_factory=None, fixed_weights=None, max_rounds=8, verbose=True):
    """Design K: the LLM sets six acquisition weights per round; code runs the round and returns a review.

    chat_factory None: the frozen default schedule decides (mock mode / frozen twin).
    fixed_weights: the same weights every round (a notebook-2 baseline expressed as a policy).
    """
    use_llm = chat_factory is not None and fixed_weights is None
    system = KNOB_SYSTEM.format(usecase=usecase, budget=J.budget)
    chat = chat_factory(system, KNOB_TOOLS) if use_llm else None
    J.model = chat.model if use_llm else ("frozen-fixed" if fixed_weights else "frozen-schedule")
    pending_text = "Begin. The landscape before any policy:\n\n" + state_block(J) + "\n\nCall set_policy for round 1."
    pending_results = None
    rounds = 0
    for _turn in range(max_rounds * 3):
        if len(J.idx) >= J.budget or rounds >= max_rounds:
            break
        if use_llm:
            txt, calls = chat.send(text=pending_text, tool_results=pending_results)
            pending_text, pending_results = None, None
            J.tokens = chat.tokens; J.model = chat.model
            if not calls:
                pending_text = "Call set_policy or finish."; continue
            c = calls[0]; name, inp = c["name"], c["input"]
        else:
            inp = dict(fixed_weights, round_size=int(fixed_weights.get("round_size", 6)), why="fixed weights") if fixed_weights else frozen_knob_schedule(J, rounds + 1)
            name, c = "set_policy", dict(id="frozen")
        if name == "finish":
            J.summary = dict(inp)
            if verbose:
                print(f"  finish after {len(J.idx)} calls: {str(inp.get('summary', ''))[:80]}")
            break
        if name == "xtb_cut":
            pending_results = [(c["id"], name, J.xtb_cut(inp.get("axis", "theta"), float(inp.get("value", 0))))]; continue
        if name != "set_policy":
            pending_results = [(c["id"], name, "unknown function; call set_policy or finish")]; continue
        review = run_round(J, inp, inp.get("round_size", 6), inp.get("why", ""))
        rounds += 1
        if verbose:
            p = J.policy_log[-1]
            print(f"  round {p['round']}: {' '.join(f'{k}={v:g}' for k, v in p['weights'].items())} n={p['round_size']} -> barrier {p['barrier']:.2f}, σ_path {p['sigma_path_after']:.1f} | {p['why'][:60]}")
        pending_results = [(c["id"], "set_policy", review)]
        if not use_llm and len(J.policy_log) >= 3 and abs(J.policy_log[-1]["barrier"] - J.policy_log[-2]["barrier"]) < 0.5 and abs(J.policy_log[-2]["barrier"] - J.policy_log[-3]["barrier"]) < 0.5:
            J.summary = dict(barrier_kcal=float(J.info["barrier_kcal"]), confidence="medium", summary="frozen schedule: flat over two rounds")
            break
    return J


def anthropic_client_or_none():
    """Backward-compatible helper: returns an Anthropic client or None (mock)."""
    f = chat_factory("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "mock")
    if f is None:
        return None
    import anthropic
    return anthropic.Anthropic()


# ----------------------------------------------------------------------------- plotting
def plot_surface(ax, E, theta, r, idx=None, title="", levels=None, cmap="viridis", vmax_kcal=120):
    Er = (E - np.nanmin(E)) * HARTREE_KCAL
    Er = np.clip(Er, 0, vmax_kcal)
    T, R = np.meshgrid(theta, r, indexing="ij")
    cs = ax.contourf(T, R, Er, levels=levels or np.linspace(0, vmax_kcal, 25), cmap=cmap)
    ax.contour(T, R, Er, levels=[10, 20, 30, 40, 50, 60], colors="w", linewidths=0.5, alpha=0.6)
    if idx is not None:
        ax.scatter([theta[i] for i, _ in idx], [r[j] for _, j in idx], s=18, c="#F0A04B", edgecolor="k", linewidth=0.4, zorder=5)
    ax.set_xlabel("theta (deg)   0 = HCN, 180 = HNC"); ax.set_ylabel("r (Angstrom)"); ax.set_title(title)
    return cs


def summarize_curves(runs, budget):
    """runs: list of hist lists (one per seed). Returns calls, median |error|, q25, q75 arrays."""
    calls = np.arange(min(len(h) for h in runs))
    err = np.array([[abs(h[k]["error"]) if h[k]["error"] is not None and np.isfinite(h[k]["error"]) else np.nan for k in calls] for h in runs])
    return np.array([h[k]["calls"] for k in calls for h in runs[:1]]), np.nanmedian(err, 0), np.nanpercentile(err, 25, 0), np.nanpercentile(err, 75, 0)


# ----------------------------------------------------------------------------- minimax path (for NB4)
def bottleneck_path(E, theta):
    """Minimax (lowest-maximum) path between the two basin minima on the grid, as a list of (i, j)."""
    import heapq
    nt, nr = E.shape
    Ew = np.where(np.isnan(E), np.inf, E)
    a, b = basin_minima(E, theta)
    best = np.full((nt, nr), np.inf); prev = {}
    best[a] = Ew[a]
    pq = [(Ew[a], a)]
    while pq:
        c, (i, j) = heapq.heappop(pq)
        if c > best[i, j]:
            continue
        if (i, j) == b:
            break
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr:
                nc = max(c, Ew[ni, nj])
                if nc < best[ni, nj]:
                    best[ni, nj] = nc; prev[(ni, nj)] = (i, j); heapq.heappush(pq, (nc, (ni, nj)))
    path = [b]
    while path[-1] != a:
        path.append(prev[path[-1]])
    return path[::-1]


class LiveOracle(GridOracle):
    """Same interface as GridOracle but calls a real calculator at the grid geometry (live GPU mode).

    Every point is computed at most once and cached (optionally on disk), so several Judge runs can share it:
    `fresh_calls` / `fresh_seconds` count only real calculator calls. A point whose SCF fails or does not
    converge is retried once from a fresh guess; if it still fails it is marked failed and returns NaN, which
    JudgeRun excludes from the surrogate and from later menus.
    """

    def __init__(self, calc: Calc, theta, r, name="skala-live", cache_path=None, accept_unconverged=False, retries=1):
        shape = (len(theta), len(r))
        super().__init__(np.full(shape, np.nan), theta, r, name)
        self.calc, self.cache_path, self.accept_unconverged, self.retries = calc, cache_path, accept_unconverged, retries
        self.conv = np.zeros(shape, bool); self.sec = np.zeros(shape); self.failed = np.zeros(shape, bool)
        self.fresh_calls, self.fresh_seconds = 0, 0.0
        if cache_path and os.path.exists(cache_path):
            d = dict(np.load(cache_path, allow_pickle=True))
            if d["E"].shape == shape:
                self.E, self.conv, self.sec, self.failed = d["E"], d["conv"], d["sec"], d["failed"]
                print(f"[{name}] cache: {int((~np.isnan(self.E)).sum())} points loaded, {int(self.failed.sum())} marked failed")

    def save(self):
        if self.cache_path:
            np.savez(self.cache_path, E=self.E, conv=self.conv, sec=self.sec, failed=self.failed, theta=self.theta, r=self.r)

    def compute(self, i, j):
        """Real calculator call for cell (i, j) unless cached; returns the energy (NaN if failed)."""
        if np.isnan(self.E[i, j]) and not self.failed[i, j]:
            ok = False
            for _attempt in range(1 + self.retries):
                if hasattr(self.calc, "reset_guess"):
                    self.calc.reset_guess()
                res = self.calc(hcn_geometry(self.theta[i], self.r[j]))
                self.fresh_calls += 1; self.fresh_seconds += res["seconds"]
                if np.isfinite(res["E"]) and (res["converged"] or self.accept_unconverged):
                    self.E[i, j], self.conv[i, j], self.sec[i, j] = res["E"], res["converged"], res["seconds"]
                    ok = True
                    break
            if not ok:
                self.failed[i, j] = True
                print(f"  [{self.name}] SCF failed at theta={self.theta[i]:.0f} r={self.r[j]:.2f}; point excluded")
            self.save()
        return float(self.E[i, j])

    def __call__(self, i, j):
        self.compute(i, j)
        return super().__call__(i, j)


# ----------------------------------------------------------------------------- other small systems (NB4)
def nh3_geometry(umbrella_deg: float, r_NH: float = 1.012):
    """NH3 with the three H at angle `umbrella_deg` below the plane (0 = planar transition state, ~22 = equilibrium)."""
    t = math.radians(umbrella_deg)
    rho, z = r_NH * math.cos(t), -r_NH * math.sin(t)
    atoms = [("N", (0.0, 0.0, 0.0))]
    for k in range(3):
        p = 2 * math.pi * k / 3
        atoms.append(("H", (rho * math.cos(p), rho * math.sin(p), z)))
    return atoms


def ch4_geometry(stretch: float, r_CH: float = 1.087):
    """Methane with one C-H bond stretched to `stretch` Angstrom; the other three fixed at tetrahedral positions."""
    v = 1 / math.sqrt(3)
    dirs = [(v, v, v), (-v, -v, v), (-v, v, -v), (v, -v, -v)]
    atoms = [("C", (0.0, 0.0, 0.0))]
    for k, d in enumerate(dirs):
        L = stretch if k == 0 else r_CH
        atoms.append(("H", (d[0] * L, d[1] * L, d[2] * L)))
    return atoms


def scan_1d(geoms, calcs: dict, path: str):
    """Evaluate each calculator on a list of geometries; checkpoint to JSON; return {name: [E,...]}."""
    out = json.load(open(path)) if os.path.exists(path) else {}
    for name, calc in calcs.items():
        rec = out.get(name, {"E": [], "converged": [], "seconds": []})
        if hasattr(calc, "reset_guess"):
            calc.reset_guess()
        for k in range(len(rec["E"]), len(geoms)):
            res = calc(geoms[k])
            rec["E"].append(res["E"]); rec["converged"].append(res["converged"]); rec["seconds"].append(res["seconds"])
            out[name] = rec
            json.dump(out, open(path, "w"))
            print(f"  [{name}] point {k+1}/{len(geoms)}  E={res['E']:.6f} conv={res['converged']} {res['seconds']:.1f}s")
    return out


In [ ]:
#@title 3b. Molecule layers: two re-labelled copies of the shared code, fpes_va.py (vinyl alcohol) and fpes_fi.py (formamide tautomer). Text and constants only; asserted.
# fpes.py above is the shared file and is NOT modified. Each copy rewrites labels, prompt text, the default basin anchors, the map scale and the
# wall cut-off, and checks that every rewrite hits exactly the expected number of places. The vinyl copy must be byte-identical to the module NB10 ran.
import hashlib
SRC0 = open("fpes.py").read()
WALL_VA = 120   # as in NB10
WALL_FI = 85    # same ratio to the scout barrier as before (HCN 80 for 49, CH3NC 150 for 92, vinyl 120 for 76; the xTB barrier here is 52)
SURFACE_OLD = """THE SURFACE: HCN <-> HNC on a rigid grid. theta (degrees): 0 = H on the carbon side (HCN),
180 = H on the nitrogen side (HNC). r (Angstrom): distance of H from the C-N centre of mass.
Expect two basins (theta near 0-15 and near 165-180), one saddle around theta 70-100 at small r,
and steep walls (E_rel above 80 kcal/mol) at small r inside the basins and at large r everywhere."""
CFG = {
 "fpes_va": dict(WALL=WALL_VA, A="enol", B="keto", RA=1.40, RB=1.40,
    SURFACE=f"""THE SURFACE: vinyl alcohol CH2=CH-OH <-> acetaldehyde CH3-CH=O, a 1,3 hydrogen shift on a rigid recipe grid. The three heavy atoms C1(H2)-C2(H)-O lie in a plane; the migrating hydrogen is placed by two numbers measured from the midpoint M of C1 and O.
theta (degrees): 0 = hydrogen on the oxygen side (enol, vinyl alcohol), 180 = on the C1 side (keto, acetaldehyde); the bridging position across the C1...O line is near theta 97.
r (Angstrom): distance of the hydrogen from M. The rest of the molecule follows by a fixed recipe (bond lengths morph from enol to keto values as the hydrogen passes the bridge; the CH2 turns and pyramidalises into CH3 as the hydrogen approaches C1).
Expect two basins (the enol well near theta 30 at r about 1.4, the keto well near theta 150 at r about 1.4, the keto well lower), one saddle near theta 90-100 at r about 1.0 (the hydrogen bridging C1 and O),
and no steep walls: the energy rises smoothly towards small r (up to about 110 kcal/mol at r 0.6 in the middle of the theta range) and the saddle itself is high (about 76 kcal/mol above the enol well). Energies above {WALL_VA} kcal/mol would be unusual.""",
    METRIC="the enol -> keto bottleneck barrier, vinyl alcohol to acetaldehyde (measured from the theta near 30 basin, which is the HIGHER of the two wells)",
    STEP="The surface is vinyl alcohol <-> acetaldehyde (1,3 hydrogen shift) on a rigid recipe grid: theta in degrees (0 = H on oxygen, enol; 180 = H on carbon C1, keto),\nr in Angstrom (distance of the migrating H from the midpoint of C1 and O).",
    RULE="Surface: vinyl alcohol <-> acetaldehyde, rigid recipe grid over theta (deg, 0=enol side, 180=keto side) and r (Angstrom, distance of the migrating H from the C1-O midpoint)."),
 "fpes_fi": dict(WALL=WALL_FI, A="amide", B="iminol", RA=1.50, RB=1.20,
    SURFACE=f"""THE SURFACE: formamide H2N-CH=O <-> formimidic acid HN=CH-OH, a 1,3 hydrogen shift (amide-iminol tautomerism) on a rigid recipe grid. All six atoms lie in a plane; the migrating hydrogen is placed by two numbers measured from the midpoint M of N and O.
theta (degrees): 0 = hydrogen on the nitrogen side (amide, formamide), 180 = on the oxygen side (iminol, formimidic acid); the bridging position across the N...O line is near theta 83.
r (Angstrom): distance of the hydrogen from M. The rest of the molecule follows by a fixed recipe (the C-N and C-O bond lengths morph from amide to iminol values as the hydrogen passes the bridge; the other hydrogen on N swings from its amide to its imine position).
Expect two basins (the amide well near theta 20 at r about 1.5, the iminol well near theta 160 at r about 1.2, the amide well lower), one saddle near theta 80-90 at r about 1.0 (the hydrogen bridging N and O),
and no steep walls: the energy rises smoothly towards small r and, in the middle of the theta range, towards large r (up to about 90 kcal/mol at r 1.8) and the saddle itself is high (about 52 kcal/mol above the amide well). Energies above {WALL_FI} kcal/mol would be unusual.""",
    METRIC="the amide -> iminol bottleneck barrier, formamide to formimidic acid (measured from the theta near 20 basin, which is the LOWER of the two wells)",
    STEP="The surface is formamide <-> formimidic acid (1,3 hydrogen shift) on a rigid recipe grid: theta in degrees (0 = H on nitrogen, amide; 180 = H on oxygen, iminol),\nr in Angstrom (distance of the migrating H from the midpoint of N and O).",
    RULE="Surface: formamide <-> formimidic acid, rigid recipe grid over theta (deg, 0=amide side, 180=iminol side) and r (Angstrom, distance of the migrating H from the N-O midpoint)."),
}
def relabel(modname, c):
    src_ = SRC0; W = c["WALL"]; A, B = c["A"], c["B"]
    def rep(old, new, n):
        nonlocal src_
        assert src_.count(old) == n, f"{modname}: patch '{old[:50]}': expected {n} hits, found {src_.count(old)}"
        src_ = src_.replace(old, new)
    rep(SURFACE_OLD, c["SURFACE"], 2)                                   # menu + knob prompts
    rep("the HCN -> HNC bottleneck barrier", c["METRIC"], 2)
    rep("The surface is HCN <-> HNC on a rigid grid: theta in degrees (0 = H on carbon, HCN; 180 = H on nitrogen, HNC),\nr in Angstrom (H distance from the C-N centre of mass).", c["STEP"], 1)
    rep("Surface: HCN <-> HNC, rigid grid over theta (deg, 0=HCN side, 180=HNC side) and r (Angstrom, H distance from the C-N centre of mass).", c["RULE"], 1)
    rep('f"BASINS   HCN θ=', f'f"BASINS   {A} θ=', 1)
    rep('f"         HNC θ=', f'f"         {B} θ=', 1)
    rep("ΔE(HNC−HCN)", f"ΔE({B}−{A})", 1)
    rep('f"{key.upper()} basin min, unsampled"', '"%s basin min, unsampled" % {"hcn": "' + A + '", "hnc": "' + B + '"}[key]', 1)
    rep('"theta (deg)   0 = HCN, 180 = HNC"', f'"theta (deg)   0 = {A}, 180 = {B}"', 1)
    rep("np.abs(r - 1.69)", f"np.abs(r - {c['RA']:.2f})", 1); rep("np.abs(r - 1.53)", f"np.abs(r - {c['RB']:.2f})", 1)   # default basin anchors (cell 10 replaces the starting design anyway)
    # wall cut-off 80 -> WALL (everywhere it is a threshold or a label), plot range 120 -> 1.5*WALL
    rep("hi=80.0", f"hi={float(W)}", 1)
    rep("'@'≥80)", f"'@'≥{W})", 1)
    rep("sampled, sad, 0, 80)", f"sampled, sad, 0, {W})", 1)
    rep("E_pred below 80", f"E_pred below {W}", 1)
    rep("Never pick E_pred above 80", f"Never pick E_pred above {W}", 1)
    rep("[E_pred above 80]", f"[E_pred above {W}]", 1)
    rep("(mu > 80)", f"(mu > {W})", 1)
    rep('m["E_pred"] < 80', f'm["E_pred"] < {W}', 1)
    rep("vmax_kcal=120", f"vmax_kcal={int(1.5*W)}", 1)
    open(modname + ".py", "w").write(src_)
    return hashlib.sha256(src_.encode()).hexdigest()
SHA = {m: relabel(m, c) for m, c in CFG.items()}
NB10_MODULE_SHA = "6655675cf4ad6401d6848d0d823bbd2e0e663ad6b87068be0b777997345e04d3"
assert SHA["fpes_va"] == NB10_MODULE_SHA, "fpes_va.py differs from the module NB10 ran: the stored NB10 agent runs would not be comparable. Do not edit cells 3 / 3b."
print("fpes_va.py: identical to the NB10 module (prompt, menu, rules), wall cut-off", WALL_VA)
print("fpes_fi.py: re-labelled for formamide <-> formimidic acid, wall cut-off", WALL_FI)


In [ ]:
#@title 3c. Hardware plan and the formamide / formimidic acid geometry (migrating H at (theta, r), rest of the molecule by a fixed recipe)
import importlib, math, fpes_va, fpes_fi
importlib.reload(fpes_va); importlib.reload(fpes_fi)
fpes = fpes_fi                                   # Part A (xTB and Skala for the new molecule) uses the formamide copy; vinyl alcohol needs no new calculation
GPU = fpes.detect_gpu(); PLAN = fpes.plan_for_hardware(GPU)
print("GPU:", GPU); print("plan:", PLAN)

def _smooth(x):
    x = min(max(x, 0.0), 1.0); return x * x * (3 - 2 * x)

def frame(s, iminol=(1.27, 1.35, 122.0), amide=(1.36, 1.21, 124.5)):
    dCN = iminol[0] + s * (amide[0] - iminol[0]); dCO = iminol[1] + s * (amide[1] - iminol[1]); al = iminol[2] + s * (amide[2] - iminol[2])
    C = np.zeros(2); N = np.array([dCN, 0.0]); a = math.radians(al)
    O = dCO * np.array([math.cos(a), math.sin(a)])
    return C, N, O

PSI0, PSI_SPAN = 20.0, 130.0      # the polar angle psi of the migrating H covers 20..150 degrees (same span as vinyl alcohol); grid theta 0 = psi 150 (N side), theta 180 = psi 20 (O side)

def amide_iminol_geometry(theta_grid, r, R_NH=(1.02, 1.01), CNH=(110.0, 120.0), R_CH=1.10):
    """Formamide H2N-CH=O -> formimidic acid HN=CH-OH, 1,3-H shift (amide-iminol tautomerism). All six atoms lie in the xy plane.
    The migrating H sits at polar position (psi, r) about M = midpoint of N...O: psi 0 = toward O (iminol side), 180 = toward N (amide side),
    90 = across the N...O line, away from C (the bridging position). The grid angle theta runs the other way, from the stable form:
    theta 0 = amide side (psi 150), theta 180 = iminol side (psi 20), psi = 20 + 130*(180 - theta)/180.
    The rest of the molecule follows by a fixed recipe: the C-N and C-O bond lengths and the N-C-O angle morph from iminol to amide values
    (smooth step between psi 60 and 120), and the spectator H on N swings from the imine position (C-N-H 110 deg) to the amide position
    (bisecting, about 120 deg) as the migrating H approaches N (weight w from the H...N distance: 0 beyond 2.0 A, 1 inside 1.2 A)."""
    psi = PSI0 + PSI_SPAN * (180.0 - theta_grid) / 180.0
    s = _smooth((psi - 60.0) / 60.0)
    C, N, O = frame(s)
    M = 0.5 * (N + O)
    ux = (O - N) / np.linalg.norm(O - N); uy = (M - C) / np.linalg.norm(M - C)
    t = math.radians(psi)
    H = M + r * (math.cos(t) * ux + math.sin(t) * uy)
    v = -((N - C) / np.linalg.norm(N - C) + (O - C) / np.linalg.norm(O - C)); v /= np.linalg.norm(v)
    HC = C + R_CH * v                                           # C-H: in plane, pointing away from both neighbours
    # spectator H on N
    dHN = float(np.linalg.norm(H - N)); w = _smooth((2.0 - dHN) / (2.0 - 1.2))
    L = R_NH[0] + w * (R_NH[1] - R_NH[0])
    u2 = (C - N) / np.linalg.norm(C - N); um = (H - N) / dHN
    ep = np.array([-u2[1], u2[0]])
    if ep @ (O - N) > 0: ep = -ep                               # in-plane perpendicular to N-C on the side away from O
    a0 = math.radians(CNH[0]); d_imine = math.cos(a0) * u2 + math.sin(a0) * ep
    d_amide = -(u2 + um); d_amide /= np.linalg.norm(d_amide)
    d = (1 - w) * d_imine + w * d_amide; d /= np.linalg.norm(d)
    HN = N + L * d
    # hard core: the migrating H is never closer than 0.95 A to a heavy atom or 0.80 A to another H (touches only grid corners)
    Hp = np.array(H, float)
    others = [(N, 0.95), (C, 0.95), (O, 0.95), (HC, 0.80), (HN, 0.80)]
    for _ in range(6):
        for P_, dmin in others:
            v_ = Hp - P_; n_ = float(np.linalg.norm(v_))
            if n_ < dmin: Hp = P_ + (v_ / n_ if n_ > 1e-9 else ux) * dmin
    p3 = lambda p: (float(p[0]), float(p[1]), 0.0)
    return [("N", p3(N)), ("C", p3(C)), ("O", p3(O)), ("H", p3(Hp)), ("H", p3(HC)), ("H", p3(HN))]


fpes_fi.hcn_geometry = amide_iminol_geometry     # compute_surface and the smoke test look this name up in the module
# sanity: bond lengths at the two wells, and the closest pair of atoms anywhere on the grid (no overlaps)
def _d(a, b): return float(np.linalg.norm(np.array(a[1]) - np.array(b[1])))
g = amide_iminol_geometry(20, 1.5)
print(f"amide well (theta 20, r 1.5): C-N {_d(g[0], g[1]):.2f}, C=O {_d(g[1], g[2]):.2f}, N-H(migrating) {_d(g[0], g[3]):.2f} A (expect 1.36 / 1.21 / 1.04)")
g = amide_iminol_geometry(160, 1.2)
print(f"iminol well (theta 160, r 1.2): C=N {_d(g[0], g[1]):.2f}, C-O {_d(g[1], g[2]):.2f}, O-H(migrated) {_d(g[2], g[3]):.2f} A (expect 1.27 / 1.35 / 0.95)")
_mind = min(min(_d(g_[i], g_[j]) for i in range(6) for j in range(i)) for g_ in (amide_iminol_geometry(t_, q_) for t_ in np.linspace(0, 180, 19) for q_ in np.linspace(0.6, 1.8, 13)))
print(f"closest atom pair anywhere on the 19 x 13 grid: {_mind:.2f} A (expect 0.95)")


In [ ]:
#@title 4. Settings: which molecules, starts, LLM, stopping rule. Add GEMINI_API_KEY (and optionally GEMINI_API_KEY_2) as Colab secrets.
RUN_VINYL = True          #@param {type:"boolean"}
RUN_FORMAMIDE = False     #@param {type:"boolean"}
# RUN_FORMAMIDE = False: no GPU needed at all; only the two missing vinyl cells (scout and random starts) are run.
SKALA_STAND_IN = False    #@param {type:"boolean"}
# True = dry run: PBE/STO-3G on the CPU replaces Skala for formamide. Files get _STANDIN in their names. NEVER report numbers from a stand-in run.
SKALA_BASIS = "def2-svp"  #@param {type:"string"}
# def2-svp on every GPU, because the stored vinyl grid of NB10 is def2-svp
SKALA_MAX_MIN = 120       #@param {type:"integer"}
SEEDS = 8                 #@param {type:"integer"}
# every arm and every kind of start uses starts 0..SEEDS-1; raise it later and rerun cells 11-16: only the new starts run
BUDGET = 24               #@param {type:"integer"}
PARALLEL = 1              #@param {type:"integer"}
# agent runs in flight at the same time (they only wait for Gemini). Lower it to 1 if you see rate-limit messages or fallback models.
RERUN_VINYL_EDGE = False  #@param {type:"boolean"}
# False: the vinyl / edge-start agent runs are the eight stored NB10 runs (same code, same starts). True: run them again.
LLM_PROVIDER = "auto"     #@param ["auto", "gemini", "anthropic", "mock"]
MODEL = ""                #@param {type:"string"}
MODEL_FAST   = "gemini-3.8-flash"   #@param {type:"string"}
MODEL_PRO    = "gemini-pro-latest"  #@param {type:"string"}
MODEL_STRONG = "gemini-4-argon"     #@param {type:"string"}
PRICE_IN  = 0.0   #@param {type:"number"}
PRICE_OUT = 0.0   #@param {type:"number"}
STOP_TOL = 0.5          #@param {type:"number"}
STOP_WINDOW = 5         #@param {type:"integer"}
STOP_MIN_CALLS = 10     #@param {type:"integer"}
WELL_TOL = 0.5          #@param {type:"number"}
# a Skala call "measures the reference well" when it lies in the reference basin (theta < 45) within WELL_TOL kcal/mol of the basin floor
USECASE = {
    "vinyl": "Estimate the enol-to-keto barrier of vinyl alcohol to acetaldehyde (the bottleneck between the two basins on this grid) to within 1 kcal/mol using as few Skala calls as possible.",   # verbatim NB10
    "formamide": "Estimate the amide-to-iminol barrier of formamide to formimidic acid (the bottleneck between the two basins on this grid) to within 1 kcal/mol using as few Skala calls as possible.",
}
CONDITIONS = ["scout", "random", "edge"]
COND_LABEL = {"scout": "scout wells in the start", "random": "four random points", "edge": "edge anchors (NB10 start)"}
MOLS = [m for m, on in (("vinyl", RUN_VINYL), ("formamide", RUN_FORMAMIDE)) if on]
assert MOLS, "switch on at least one molecule"
MODS = {"vinyl": fpes_va, "formamide": fpes_fi}
N_THETA, N_R = 19, 13
BASIS = "sto-3g" if SKALA_STAND_IN else (SKALA_BASIS or "def2-svp")
theta, r = fpes.make_grid(N_THETA, N_R, r_range=(0.6, 1.8))   # r = distance of the migrating H from the midpoint of the two end atoms, Angstrom (both molecules)
make_chat = {}
for name_, mod_ in MODS.items():
    mod_.set_gemini_models(MODEL_FAST, MODEL_PRO, MODEL_STRONG)
    if PRICE_IN > 0 or PRICE_OUT > 0: mod_.PRICE[MODEL or MODEL_FAST] = (PRICE_IN, PRICE_OUT)
    make_chat[name_] = mod_.chat_factory(LLM_PROVIDER, MODEL or None)
MOCK = make_chat["vinyl"] is None
TAG_S = "_STANDIN" if SKALA_STAND_IN else ""; TAG = TAG_S + ("_MOCK" if MOCK else "")
if RUN_FORMAMIDE and not GPU and not SKALA_STAND_IN: print("No GPU detected: fine for Part B; Part A (cells 7-8) needs a GPU runtime, or set RUN_FORMAMIDE = False for the vinyl-only run.")
print(f"molecules {MOLS}, starts {SEEDS} x {CONDITIONS}, budget {BUDGET}, agent runs in parallel {PARALLEL}, mock LLM: {MOCK}, stop rule tol {STOP_TOL} / window {STOP_WINDOW} / min {STOP_MIN_CALLS}")
if MOCK: print("MOCK mode: the 'agent' is the frozen twin, so every agent result below is a pipeline check, not a result.")
if SKALA_STAND_IN: print("*** DRY RUN: PBE/STO-3G stands in for Skala on formamide. Do not report these numbers. ***")


In [ ]:
#@title 5. Vinyl alcohol: the stored NB10 data (Skala and xTB grids from the L4 run of 2026-11-12, and the eight agent runs with the edge start)
import base64, zlib, io
_GRID_B64 = """
UEsDBC0AAAAIAAAAIQBU6m00//////////8LABQARV9za2FsYS5ucHkBABAAOAgAAAAAAACSBgAAAAAAAJ3V+T/UeRwHcA21jpo8HCnqsbLVkKMwlVCEEEu66FBhDHIzoprBJjk6sDlCjFLrCh2UFBmU+2jFoAeS
SqnJUTJlJ+3rs3/C+uX5+Bzv9/vx8fl835Nsv8/O4cA8sVAxNs2dyWIE0wxVacYeBjQtVZpHQHBIsKv/0YBgdyaZt3T1ZTExz/JyDWRirK67WUtVV19DSzVc9X/+SQtHct0NtBm8DuvWMaEOg0dT3zXks57BmxNt
9yjTZfCoEalnavQYvKbuhhxPfQbvfkaBUQZs2vTNJhTSs+kz66DtA86h99jnNrG7sxhKXxxWi4T3j2xZdRhK1vckBCK/QpO8OAX5d/lYlGxC/qMu6s4bsP7KsjHsK5QzVlPwRj7ZSz7CNLi29NmNJChqiFP0J+NU
ryumsLQztVcODjieshxDnCwlnvMU8k+qxSxbx+DpGJpsSkCdprZAnyuoo3VmzfpjWPcLCM0RQ9xI8L5FLlC69WRMAtTq+SBPzqWRnHfwIowwOrLuOAzyNb7rBDnn9CTpMCXobnEX8tzW5jKYqPNEpb2kHXVaymm6
DahDYYbkhGI9+FDvRSGk5keF2SHu6jalhZEwc4nn7lS4hx9Snw6j5Ysbz8GWXfn55lBn1yy1DXFx5b0hdDgUvmqxN+rICkoFXajjVZ3Pb0adEVmFy35Y9/6a7PQKZoRymvUQb6+9QtsHFhnepLIgxTeHtQMWcnRL
p7GvoafQn9zLXb8d8TPkfpmdf7jBWccqnj3qLOiYHSxHnZqooN33MN/Q/9HVCfs/SwasKYdZl08lDcOrTn1NL+EJ9Ym996DB5PZRL/g863ucBKwS5gQlIr5rj/WyxVCOUzMQj7yjq9c7z+AdLFbzGDDDOIJrHmOA
9WsUx4l2sq++3ZmK+LTayVhFSIu7JBRiXiIuxbsajnsabfGDF854WS2CPYGRi3KRZ8F57pQubJYvzqvFOaa2FXaT92ZZ5toairG9zvjZbViXL4rXbIXvHivzqORdTGjekoPDkiEdAsxfqNvcchs+PCbL9oT0DUHl
ClBwfka8CnnyymZbXSCV3URZAIuGZY45oM7OTAnxH9Cn3/1SEOZ9TQ9/r4Qugwl6I3C5iOkigKaPloXxIeNEREwB1Ke7z/lCsamkmrXwcbGD7VvkKbrfFcCFCSZzvk4wNP7xuwFoVupiysK+42bv85qhyoMfya/h
xjTlV40kPyW/JQ723julbAz3OrdYCxBnUx3z2zU4+dm2ggndFG3G9eFCK6N/fuL7Z5dFKLVD/mm2RDbiGuxsI21x7uPDy63qYa726nRJ/J8E32onlaDCmwLTD5hvc5ea5MAia8sTg4gzGtDI0yDn93toHYn8oxXl
ZtPIW2fBVk2H3RMRlftWMXiHLQazyH0EHHhjsxD5Ynv9v+bCvP73vdK4d84vxlwrKPpp1HEAHhEb2m8ITz8QfenHPmfGoiId6MVSXmqBPGrjx9W2oK6Wg8lNa9SlmG1NvIB6Ab+eCkvDemktvZaN/VHiJe76yLM2
KyvxMdzv5Cajgu9Dsi0+mXwnxhE1VeR7TXieuV4Tds6JCvqw73KMWMVWuG1zeLEt8lyXkfGQRV59DedNDaj34GReSyg5T2DiPCbWz8blCEhfu1+h1z4Ktf60UiD9q8/b4/RTUm+lg8ZHKPclevkIDEnKfnkbfriS
63YQMvw/6TWS+AO3mp4hn5k+MzEI+fl1ygI7WKA9pbkD802x4gWLsa+cwpL8C1p3U4eoiK8fax/LgyxHqRIpOvqLwvlhXfh3oVKLGvTN5lPfYn2paCUtivTDnVn+pI9+X1hXPYq8Z9aUWBqhznLhsQu/YxyTU0Zb
jXVWYZgl6bOl7SJPXcSp+2yQCIFDk311l6FF3zRFRH4n5h+t3Ig6SeZseU1IWfmDO4p538pHjSdg193vK16QPmdVR59P3pfVD3M91FkSb5i8CvPmVUln+fBR/kQGuZeD6mOx8dDRIcXxBmx+OxdFzhenEviK9G11
9PH/6kalRouhXpWhzkQVOV+0mAqJn03Z0MPF+xh/39z/CfV20IZvkT41vbEivRuuMAiXIn0+QbB3OzlH7rhRTz6s5td1ZEKqjXJiONzqbX99P+mv0eUpEeT+pD1uXYXWaRyhB1Q1yWV/wXu4t/LlBy/U0fmUMXoH
50qwGJYl/fMOQ1uG1HnXyW3lwvHX4dXXYXaqyYvT8MmLqULy/ihtilkKsCB9RmkQcTaeV2wrocOS6ABJzP8LUEsDBC0AAAAIAAAAIQAwiiJw//////////8JABQARV94dGIubnB5AQAQADgIAAAAAAAAQwcAAAAA
AACdVfk/1ekXJ9ukiSE11u9YnkmT28gyZb0tlouRNDdkSXaFlCvbIFtfFSJJGsqWEdky2iyjB3FtDaaFXNUkshMiVN/m3b/wvb88r+f5nHPe57zP+5ybZmWzx9qeny+YL1zVwzPQnaOqp6hq4KWjylBU9TrGOcFx
9T90jOPh+eXd1NUv0BPvgYddj3vivklDl6GoofUDQzFS8f/8iV7QHhJe605ojs+a8zqhhEq4zCxVnyF0vdinwd8yCA2rNP47t4BQuxWPDwllhD5QWLjddJPQOi/ekE0FoUxZeeHLJYSuOLUFulwn1IXffjjtGqHJ
zjazQbmEXoqLKePPJvTAy7mBvQGIX1RwejqO0KYngX6vUwl9Pft55OFVQnMlo4w94H8xQmioHDg/VWZujgSOiou5cRJOpaI62alyQsMnQw3jSwk9X2gWMllM6D/Zayb84Se4NuOF+u+EVm8b7PJDHSqxmh8FEghl
TUs5/XqJ0IWgt7r7kc/jVcWrlmFv4aP9tQJwxsd0bAdRh9S25upW4KQeZWkcwZk/LTrMwvuFtW13HsBuK2+wsA11phq+SKssJFTTZ/qrteGEdtwve3YxidCv5BZXAi8T2mNoe/ATcJh+sULRwGkUdH+0DvnqVGnM
jCH/a0zzBUfEd/Uv1ur4glNvMqWO8xv7nrulsNOKF1jqR/zF4MrF+BxCvQvCSjb+Cr6bTQZYwIlWV/aIAY6R4NjWROBMslRXO8B+a6M4I/cG6ov9I+Eq8rUPPhStAzzJIrt3nogrF5UgFlEEvr7rOjWfT2j7rhSe
6RVCRxfM7z+9SGjbHX1j+2BC091vndE8S2htxh6XSPD21o8XUYX+eaucrnBGX1OXSoq44FmPbSsmBdzBumsML+hjb73LsGAeoSGmS0wTxA1XcGwIhv+liPiu+RRCP7d7lYsjrvcWs6k3Rwm9eaO2STOa0OahzY4z
5wg9M1r5yAP2fy7L9uplgq+6xBcbsgit/5ptq4Q7r0//aT702KK5Wrk/jdCCk5ffSiHuOrtzNTmIy046z26ArhYbvm27G0no09f5N2tcod+JDgkZ6C7Rfy5yAu9l3DschdPQi8y+g/LgM6D1T3sn4KfzbzSTw/2X
EynDEYintluasfhfQrdceTXyIIbQ5/1nnyvDv+IvpTYj6Mvy9FAaO5BQm4GhLF97QqcaeW4qHoRyVxhuBqivO8KxIfAEoVcy2t8nhBEq4j0YxYogtI+Xs1CHc0bG38EW+mFKPzOvQjzdzsIV3yDo6MGcUT/ybd4x
GaXoT+i9Ys68ow+hHxYO8Qs4E6raYikS4kXoUZdG3TE/QtVv+r4Sg/0e1q1wyiHUM0MlYQdO/oCGd2HHMYeONsmnEKfLRsV8EvY3FLkv3XyhJ9H2dawjhDr8bE8rEM9o03plYeRv29o96wG87qIeYzXkn20nE1yC
PPcemwxuR7+Kix92GoBnB8EOdkwsoTUtrwR+OUloZl/VfmYIoQ83SbbOAi9UXKaL7zBwthbKySJuk+Rye5wb+jtytcQCfZlOTNLMQb0WRrsN9oLfl4Zq/0vEvGpfb2Hvxl54b7U4FpdOKN+uqXhH9N36qeT1KOgj
Mb1PXvsC6qiVE+OiT8zvlZQL0Rf3HgbbEvWmJ3GzxVDPvvcjW0ZcCB0wNlUyBd/7Q1vYZ2HPqOm1SIDeY1TzVQSxd2p9j0R3YR6KTEZ3zkDXzpW8pAzoOvTZDxViuP84nq03hzlbkvc88A5z5zKxXeJJInjJHEgo
wFwqmN1aXQu+rPY3rBkHT06m/QrlwLmSdCZAHPlGKgc9vAz/7vKiiHuYvzVNI8kCmEv14xIqzpj73iWBziXsmbHfdfr1MbcOwo1zHNhtCLZ15cCP8TFF9zHqVTSzcpOPItT3M4+1CD2J75yNDcOezpd47+OFehbV
ik0uop42rvWEPubRV15KawfiPf/QNv/xD+jh4LEI7duEvkkqK1S7RaioVggzCPjq0n491dgb3fyaWZrgYbPMhsZKzBdb58nRcejSr6u+sRQ4Nt/7mtxGP6qdQhwzsJec37FDOeBp7i++3Vz437fo6tQG3mEHzcVl
4MWnWfWvukPo1WFJ/zHgMfrsBnKAF5UiVKoEe6cukYKN2Be//ahfPQod2K7oN/4Hc6nbYaFbChzDdX8LlaL+ypMZxfzgY1Kep6eLPfbIp6lZCfwJs7V9onCWHhROHQdutYHyYckqQtNEX09fA36vU/KIHr57cFPN
WdjDn/VEo/ZBH8KrtivLQ7/OrdzOWtSpYeMpfQ44zRlZOf8ApzGr95Ma4hkHrJyMhr+S9/PcRdyZRqODX/Zo/cdt+xyR/3GFkMyfcBectxZl4TvLujJvJ/6P7q6XsGF6oz8zyd9EQA9DlmlvapKxH0SE5E5hjwq2
X7j3BedxSGTqbfhZ7zqi6gGc7edkC/MQ742dv0g94kTmmVjEgd8cPcvvDmDv+gVI39MBX8thUnnX0d9/AVBLAwQtAAAACAAAACEA9dhgR///////////CQAUAHRoZXRhLm5weQEAEAAYAQAAAAAAAHwAAAAAAAAA
m+wX6hsQychQxlCtnpJanFykbqWgbpNmoa6joJ6WX1RSlJgXn1+UkgoSd0vMKU4FihdnJBakAvkahpY6mjoKtQrkAy4GFKDiAKFNoLQdlHaB0p5Q2g9CNwRC+SFQfhiUHwnlR0P5cRDaIQEqngihDyRB5VOg8qlQ
+TQHAFBLAwQtAAAACAAAACEATZY9Uv//////////BQAUAHIubnB5AQAQAOgAAAAAAAAAfgAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBuk2ahrqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGxjqa
Ogq1CuQDLmMweGyfBgbP7GfNBIGX9mfPgMAbewYw+AAV/2hvAlb/2f4cWP6LfTpY3zeouh9QdT+h6n5DzfljDwBQSwMELQAAAAgAAAAhAIhP4HL//////////wsAFABzZWNvbmRzLm5weQEAEAA4CAAAAAAAAJoE
AAAAAAAAnVX9T1V1HD5iBggmAQkU0CHEK5O1Q5gDAXdKoCzB49vQrsGl6yXe5N6AgGBt17kAheQUZm2ZOzRaFjIPaM4fSq9jqYHTSyO1Ky9XIgttdQk0BjXjPs/6Bzq/fPb9vD6f19OWsz1b2blAqBbqDbstleYK
w1rRkFaYbEgQDYXWiqqKgvJ8a8Vui5efVVBWaZnnVxYV2Czz75WJKQliYlJ8gvi2+D+/xYIgyO3qE/I8tc8WhXup/HBPsJcKm66DL9buCvNSZeSTx7zU80YLqGPfiRAvdT9kWual6pN7Qd3L34FcnWoL9VL9ihN6
Wtkc5J6LxxEnKLUVfrWxXvCV6eTHgWMgIwJ6R7+GvSdbZ9yhTsb97Rio2DcKv8LHi6Hv/sACfbHbB28hvB9xpIHz8C+4bsLOOamCL7Rew1u5Mwo7d9kg7dvcyF9b0UP93AHwndEjiOdIrgNu56IM1MfTPUM/Ey7W
aWiauP0W0n7CB/7kg7/S/5kfiX+Vi/kkzbFOnVeIc//7wKdNj0EeVGAH/0z7EhF2ui/jnfsLVLrmC7l84gLi676T4Ntzu/EWpsJYp+9nGf94H/SlqyOQi4Mj7NfAetTfUTYDvj1lHDjcz7JvQcda2Zfgz8HXS8LA
12bvsv+HfgYVxy5Bz3RqDn7dDxLp/6s/QU1TnBN7XjXnYel30Nfsz0AurLqD+EFVVaivcjmQOHPoX73RDX09NBRv/Zd+yD0z06AO42fIQ+5aEwm7rEjOge9J+NMm69j/jFTURZcMxLnWD/amdBfkStwAqMfcRByZ
H4LuCIiKBY6lEuop13awX8MriNMWAnxyeDDrEHgZdkH+64i3xB98R9zfsHPcSoOdHBDD/gW7IBd7oljPQ8ugJ367B3kpeenAqxmIV4+JZv/Tn8PbfWs1qHrezP5s+wl2wqdnUQ/pVCzjNf2DOJIhle+h39fAX0MH
8b+exb3ry+Q89YjQk+4ncK525nCO4r9Efnr8C+xzej5xdLzEOl/P5BxtTEH8G0c2p8FfTTNwaadl9EHZUBoFfvNTnI/8B7wTg8nE0bgVe6Q/H0X/UQbmF7ec9R83su/bIoHHJJdyT2t6md/sLvbL/xzkNs8W7o/Z
ynqWMh/ngpeJ994O7mVcDudqXzb8Sx9ZuQ+rG9nvCwujgX+vk3N72wQ729lN0DOZi3kXZQfwKxGvcA5qq+hntor1DdzDfIq3E3dSHeqiNlk53/cP8r40BXA+3ryIesmX3kU8bbiW+/FaF+LYBi2so18D5YmjsLOH
G9mngHres64NvFMNi5Cf6YcY0sib3MuiI4gjpcRwDpfkc0+t5bxXxi2874pC3CNPM58w1tfeboCdfdjG/U8hTtPWleA7J/6gnd84+xZ/kvv43mbWvTCPdY6V+Z9ofPS/e8F+ra9AfmpLMeMVU09Xy+lnSOf8O0I4
J54S/nf6mjnPsS8yfv9R6OvGXNr7HEYd3TXrkL9p2Mb/gP1V9vdqJ/zq9+7Cr1p5G7jU3hbOSS7vnHTgMOc4V+YdeisL8Wxf1POOm/fzLts2Qt92WsNbeKSI8ibOhVBdyvy+CY+Q/wVQSwMELQAAAAgAAAAhAJVz
hmf//////////w0AFABjb252ZXJnZWQubnB5AQAQAHcBAAAAAAAATwAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBek2SorqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGljoKhsaaOgq1CmQCLsZB
BRjoZREAUEsBAi0DLQAAAAgAAAAhAFTqbTSSBgAAOAgAAAsAAAAAAAAAAAAAAIABAAAAAEVfc2thbGEubnB5UEsBAi0DLQAAAAgAAAAhADCKInBDBwAAOAgAAAkAAAAAAAAAAAAAAIABzwYAAEVfeHRiLm5weVBL
AQItAy0AAAAIAAAAIQD12GBHfAAAABgBAAAJAAAAAAAAAAAAAACAAU0OAAB0aGV0YS5ucHlQSwECLQMtAAAACAAAACEATZY9Un4AAADoAAAABQAAAAAAAAAAAAAAgAEEDwAAci5ucHlQSwECLQMtAAAACAAAACEA
iE/gcpoEAAA4CAAACwAAAAAAAAAAAAAAgAG5DwAAc2Vjb25kcy5ucHlQSwECLQMtAAAACAAAACEAlXOGZ08AAAB3AQAADQAAAAAAAAAAAAAAgAGQFAAAY29udmVyZ2VkLm5weVBLBQYAAAAABgAGAE4BAAAeFQAA
AAA=
"""
_RUNS_B64 = """
eNrtXetv20iS/1cIf0pwMa/fjwsywM4gdzjcYi64mW+zg4CWaJsbWTJEahLfYP/3q2q+uvkQJflFZW8fsGJTzerq6uqqXz36tz8v8jRdXvxbRN5FF3ebZbqCzxc36V22zi55bC6vV0l+ewF/vM3yAv72258Xi2S1
yuGjgN9eJdttlm7hX5rESlijuOFUE6ngj+l2u8E/XYqYUakJp5JrxakQ/3gXtePIcBweE8uFMUYKoYmx/kA01poRbSgn1mp4wgQjqWAkZWMgRXEtqeaUW+6PJGMptGVWKENhGBaMo7sz40xyqSw3WjEazkwKwigh
QmmilNAyGMiEA7FYA/FKWmuM5iIgiMXAIk2Ixb9LTlUwkA0HErFhVhFCCScKvuIPRGLCpVXKGCaIhvmTcHKUPOFY9AnHYo8b63d4YJkusjzbrPNSUovbtEhQuGOc89Z9wrX9evuAYv6f66zIklWUF0mRxtF/ry/v
k+I2WiTrZbaEX0V/2xF+ZT7Al7cf4Jskuk3yaJVsb9K8cH9c8H+5Se6jNzLWb6PidpvCt9bZ+iZyA8FT2R2Oc59uo5q0aLtbpRGLL9zkawrZCIm/JHf3KxzwDfwZSXgbbdM/0mSVLktCom+//nj5H58iICOOfmoo
56OkMhXLtxHQgSS+exTRtiWauk8V0X9N4J332eIL0HqdrYHU/A4W+XIbpevNKkqXN8DtT/j3PFkuYeCbXZrnUVJEbywMA2PBNL9mQExJ+Ace66jYRMl6cbvZAskpDJwsCiTsalMUq3SdLr50iOM+cYIE/6E+gx0F
2/QGR7vP1kDu+2i9+Qq0AevTaLcuPyxL4q+SPFtHqB7vdneOZO5IFkBySyLMO92m60UKX0q3Nw/RNfyykm2Qr9Vit0qQ/A7NxpcC2xL5sX1zRUxJIvIYF27Tk9x1mmxr7iKRxomPrfl6m93cRnl2c5cg0eUqeayM
blN4oBinjsYeCz9tN1e4xj4n35evhOfglVneUOjeXr5YxHCAoEjq2IRyuNhtgXlFw7BaJDv0gNadkr/FZn2dbe+AOo+cq3TllhfJLflY4Btwk8GQtfwly78nC6QCGFRNDT79kW6z6wcngjV1MLv1pnAsxR2z+brO
C5jL3T7uecT+1JLovR2O3cJfxPcNjTUnqwk1W6XRBo6nuKq4ZTe7hrJgp6CyzJbfUE3+BuMY+Pdv1NQfVPUBRJDjT3wT/mTVT/i9wJ+8es5Uz8FP6QYg1QMGf+K7gEOpU8r4j7t0vfu82ty05kRlBWTLyqwoEvzj
BczmbzvGFAEFV2ksNEVmrdaBjs017n6cy28XvobDv90l3y7LV/gz8n6brB++wnKl+OuR+fu/9h+/BnG53m7uKiXh3hcqrXetOrv4vTl9K8vJcZ8fzP1zObI6CzKfBUDmwew3d+FS6HopaLsUXSma6wl8NsI/wntT
8161vN87yJlZG+e2F2y9HrpdDzi8P5fz9BdgdqZTh9Uh1c/8r4CHzt9zTJQtE3cgLdsiydbFGBdfy8Tr8K1D6Mv8M+QfrflnJvk3e5N0ftxlA9yd3OKvb0G/7vZGK/p+s8oWD7Udjb/Jd3d3yRZ58mcNpHz+Anyu
0BR4wgnfEo8KZBxO6yL43sWvt6l/8rxxvP/AnalcHjugJ6MCdHWeudPfGdn1c7Z6Dlj6LkpWG2C24ywsNSjY7TJbg1m+RgV5tdmCxrh2UluuD3xhm9Y6GrRG9geuVRKB5tnlraKA7+BcIpzXv95tVs7Luig2X1IH
+vx5ka3vdwhUKmKR7AtYv/IXXBmKiBMwIS8+73JcuJ+Tn3H+KfBliV/nTMeCWUQENTNWcCeqFURKHwuRGh5zyqmmSjIluPKhTRMLoRURSmmqRf3mUYjUUCuMkUAlVyHWCtwXimjDrWGcUcP2QaQ4kqTMSm6E1Frp
zkhUMKCWEGMpDLkXJMWRLAPTxyhmiRSdkYjiTBlrKQxDJN+Hkj4GRDwTnFTGyAQmpaQEltJfQBJzbrU0EtaVckasmEJJJVdaKqKoJbpDFKWwclxbpg0R+N99KOmYM9Hxp3/peAyVN93ufTx+WjvW9yJorN6XWxz3
dmvATiFP4bHKQ/Oa98zr0I0BxYlqutEg8FoU1rgm3J0ZSDknPTPm4+d7+OoHrWIb/dAYhaABqwkJdHx9u+a4mbBwJmzIXqjPNA5TaW2HejIwVZiM7U+G9ScDp9oHymIeUuzshkeCqH130+MDEhBHH6fcmVGhETF9
H93hGVD5SJkzgJaV5V0xAoZOQarSA1G3AXKus2+1c7C4TUvvoFrx/HazWy1LdvtWA/o6jYw4F7libN5hKwjhwrlgTniOQlcbl/EucZwNhLtv1NxvwMbyTUvPtCktcepoqKkGyxxohUHvkodom2R52iW94nCHXO2T
q4ZE4Ut6X3h+FGqq9y1n+37XG+1oVjXNtQvBHL017Y7WBS4HWBVFug3sXidqo4GVfZrDl+B8s/oDjcmGHk9ndfzzrylYNr48s2avenOAPdcBT2qCu577FDILP5X7BRyvlPlQLJ5Wuv6gOuCs8MFZ0FiUVGiuPhac
fQQm9SrHyLkDUifgsmd9WJ4tWHvq+szRBDi3RXhu1HZW5s3ZQrheXG+Z5cnNNk3hrCkOQXhmYa11GN+dw4v+ewTgPVwJzd/afAY5H5LnLylstmPkuYWD2cHcnrGxfBCbn0BvHM9nNhC2OEirn5vJ/1JLcJjqnoaW
x+HiYbYMIsb5kPWJCDDogttomW3TRbF6iIIT85cvySqJI/fSOuSPOQMw4lW2yv63eg4IIbFoEOJo80cl+StccenWPK8Wqzox3JoR2DEtrDyKlveRdTsCQQvBRABBM0Y5n4CgqbUxUVY4RFYY6yPQ7NFJuhy0tCJC
a43/193kWmkltQwescRKrvZh0Cy2QjPOhFI4YDiQpYZzIakxSpAOlt1DoAknmKYrKWdcdigyXHJLFBcURpNqLwItgX4YxhpDlRFcBqCqsURbZqxiMAwhagKAVlIqZY0UyuhwILIPb+YxN1QozqUg0nYwcGEEJRzz
nzHBeS/WLGNmBaPMcC2JkSagQDDgqjSSEW6BP3IqI/fQuQxAywd/lZ/+VXHUVx+LX/vAwa2fb9FNhOlGtFCdwx7NQZNl62mcVx6VEALvq7Rg9BUUWuPkebkhfYrDI7VRvPDqfooIfvN+A7au40SVJpLvttfJoguZ
Huad/he+3J2F5RtKW/GSgifaKNHxQ7RPtaOZlTSDbR8cmSPY7tHeW/Ce/Da7LjpIh0Sr63CaD8nFOTDDNySyHXgIj4EDx6MSj6VLL7w+ZJuX6SurNFlWgFkbdt2X0j3IxJqi5gQuISIxyLiR7DJasuwe+InJCX3r
YJB37Chcq8qxAr+7l2bV30qTNgxQm6KZscryW/fdYrtLpxabHmuIdmXy6iGwZFr/q8kEaU1SzzegPtmoSMCQQl4XtbXW0l3NtkO4OIzwXytqSlVpSK0pXdZFK2JOc6Irgy54cZt28gedsHr6NP22AGsnd3IehvmP
Dwh2HCwfMnO7qPGwFl4aaqP8fbCzRA8wOQTNWbC2a3+8na03h22GGZZTadaySptm1c8Gy5fVEwNQfv2gbBB8/EL1U3gxgFeD9p/vhD0czD8ADDgdiqdTiUrndfjPJlFRnwrjztli+X8sfV7W2PeQAT2dvDsX23J+
6abk1OPu9Y3hV4xlHy22LUqunge9fVUr/9yUSIulT9suZ+SzzKfEgh9cpzJH12o+fBTPl+7yyq7gK6sMLC/5eHrIZyDmcWg1QSWooKFrNey5ftEyvWncv794AbQPsKhVpKevtIfG8oZiMFQc/VgtMAaImnB2XoCd
8lAvehsiShbbTZ63QSJRBolGiwu0FlIGxQWMazUR2WGGxVpQRUGYuNFSB9UF/AliO5pqzeE/ygqmOlUBVGgDfxFGgsQYw6dasIAnyRkVBKsgui1YBFFca0ws5/DGfdGdxyTN66cbyjxba5EpSCT9dr/abF02iWcU
HgKFVFHpsuDQN8LL53t24mnoOKqxxa7IMWvma7nZ6m1Wa7OgAChAd2AjttoSSHZ5M+mQ5YTH/YGp2VWtodObjir4h+NBBVw07DKkNS6y65FzzoxBiHY0A6aaHjg3CMm0CXetJsPiOiADCWiODYuPU+JV5DU2piuE
2jaE46sPgAZphfzxkY4LokECXx7oe0mpflLU78kcHDWQnTJs8c1xI84P86PH1Sa/iL7Yg14MY2bTGM0ra5154DGnVq8OJR55jnx3h5SJRwM6xUs+6qmo2+SPKgfpKk3X+xORPiEKlvsC1gxTvdobGxYJsSJYl3zT
lrlW69y6emH9Q8eXKNe05+INpETBuCAPN6WFHIh1aGSLyYynsfJaYTgLcpsomJVTuU2GxlJwqTBJR4IVKX0LWDzaAmYx10JQMO244YqG7fWEpWADc0OxEhWI2G8AC64IlcxwbMrXMaUFs1JSTbC4UhmmJgzgQzNh
evauUIxaeJelFgzR0EgVhMNcCVNMgwFL+SPymR6VUpPevw/P/U6We5la57Jem/PKk+hmf3lVPLXjNpRXc1iopwIHm4Yfwz5kSynuVq93wl2a5DuEdZ0XnRWn5SF8bGAu1N3NEQ72AGhlGmDE5R9Y+Qfum0l9yk9t
c+dqC7zhWnuhOSFqynwLAq0HP+V10nLldWUadhFTvu1aR7PrqLWzZV8jSP38MvvdlJ09Tyj2yffn2ZaNsSnv4bWVzWw8BfPUDtcTqsS5N4OZBGp/6RnEHWQWd+Bw+GwQfh3P5H8XPWTpaln2dmmtVBz7sthcVmkt
fq+XTqTiiPR7Sbj2TVTKzCRIS6mNtWv3TBk3TBLtm6jy0SaqjKkAU0xSsE8J02FmNzPY/5lKTHhH9HWfiSpiME/B/qRKdTqIgHWoKVOSGGoUWL2S7TdQuQbD0BJhCVGKd8xMIzXRliu0Ram1an8LGAvmt2TwTsKF
EuFQYCyDAS6sEFxQouhMW8A8whp2kuHcryF0q25DVXeeOy4xrkkPaMq30qTbrUP6CSBqotK7X0jsRox+3tscz6tcjX6IaK+MeKRV7xjs8T9VPS3os8tt01cuCOHXneL2ZE+UATmf3XAIIooyAv0e16yxZVTAwJJT
J6YnLdMiRa+/Kn+sssX89o1eNtmBuHkFINYQ2AAOhHS/rw42U3a9LQszUS2pKLnagBHQ4Xe9oO9aTQ7Kbv1lsHXMfs+A1Y0nSP2h9gxAtRE/rzVAuV/eQ3iFffzdOAwe+4eJnqWy+qfpEfwauve7aQosn7Up8AzO
mafaBs+XBWtODhO9/vF4LsGXvX5hEOFoM/kGwy+1D1gtbhz9e82a+yMjJ9Xe8DPf/LBIMRC5O7R424x4j+AYaBF4j4pMF28LFhtslakZeHJMhyk+6glSfCz2heQc/D4SAPmXmIcNHhxXCrw1yrDX494Ah6TSuBQf
yrTD5rwABxdWg1+p0FPTTE/UbzOLXh3Ham9w/8KRpAY6GRZEuzub9ib4AOu0NMr1vSSqk3RkjdWECCaoYmLikiWFV0MxeBSYIVnnkiWlNRUYJSKMMb3Pc2Sx5lpZ8AqBH9R0b2sC510T9L3hGdqtwia9YnkNRBFN
DLzedJhkwMk2imlYOmXJ/npuYJJSRDELDjY3phOZstZKIgi4/swaI/ZfsMRiJiVBsAMkXYa9Q4GJlAjLNZPYYlZ3Akzdem8YimlYHAPzM0x1h8KieEZhfkYRLfaWfwNjCVUChJhxTWU4EHjoFjaWBEkCwjpXZNFe
gwLCqODAKEGwXj4cyoAEAMcFhY1CdQhGUNUbScIAQBU8yU2HKNjv1hAJ5EphbCcaSHWfKMM47lNLOgFKw7VL3TMSZLTTIpd2hByXWmpOhKbw2u5AyhqOEBNIOsy8M1JfyqVUzAFSDCS9I+XEWgECizeyCcLCoRg5
OdbJ+g1xhdIKdggwmpHwqyA+jFBsQkw5TDmUasaeKth5aGO2TrVjWBPTpAvXWKh/qYm7zQT7rzVH1dt+J7aOZXFi5eunskKn6hzHRlv7SS9Tf6DNkNd5rvGHm8Zl0u+vvq+l3NhkzLE1Yz49w/NpWTt080/VqUpU
02lmwoIuVRLdpD2zmbiXKuj2JPeVDVUDvR9v8CQrStuWWjYWjtgySIkTdovQ0ORM+5D8wUr/6Z5UeYozXnqktIR6/adcB54qYiq99lMyJKwMjY42gDxA3INbhsKitpaa4f6IoOQG5LyUpvKKIpxFyV7psVcgZ46X
a+Xz1owR7b09Wf59lxc9YSaDkqHcZE1PMiSYwL5kyL2kD/TK5SOQTEh2+/peIQXDQooBirmbqPKUCGpDR2d5w5OJzdtjqaXyIHLbdw/VmIIuHtZ7copkG3MguQydNjdRea5Wl9ogkKsP6PrZlHPvo54NU1+WtOlx
6in4WgMch90D7C7q+YyrDOKL9aeG3Pa9Xk+uUELoqLYjjUx7d+y8EbF922iVN8pxva/chsg1k+Wb3muH2ettQK/JYbvt2pvMXLAyJmPqrYTXujKhJiXYe2uw27xCloClb6gKBLe2PEpm3q92ec1HEkhvWDI2LsZm
8n4bn4JWGbu4bo0xdhHbwYtYQnl1BUfw6Qf+1st17UEB42GnPRXRZgBm9loxOinokWzDW6GQSBHT6IcoyKhwzy6zfLFN75P14uGkbLZ+rWVQbsywTL4fjJjqWhDe7sYGYxMjjZcOzElxidl5dfnfQPNEL02lzkYh
3h1bnhC28yub47a10mGB2Ai99DA+fxxs70gbjo228OW4AxuENsPFWuctgb3Oj4d2efd7u6ugyXv9JBhENujpTuurOlWdhser2Bqtg2wuDkerb7Hg2k5VPeJVl/ChzD2K/z4yLvf0DbJf2g87tyDRM/aOn6t3+U/Q
Pv678ZnPtjPOiX2f54QEzKvl8FCg7ymba78SkDFbJrdNcNhJJ8OsgZj5cv2ExvHnAyTNl+3s6NSk2eNg82U2P5rZs4fx5stscTyzzxOFPLt+W/LolZk/tnp2i6Ce7lqc14SMz47v+ni1dBZA+NkthHnkBpgjvn92
i2CP3g0zjFrMNjGYkeetVJ5D7OXcRJ7Rg7thnkEkaTYF0IydKurzi3+dhUgflSn/Iya6FwNdidpdf0A7Iq/kqWxENNSD6F3JfJ+1rmnxQ6/rj33a+8ykVq6rZJMSj61xxERKvCKYbKsoVdpQxkVQT60fnREP9hth
linLpDJM+knaYC1yzaW1gjD41E2Hlt0MVK6oMYIzzYwKMuJJLKziVHNBDebDC/FcHX8O/uaT3VgmY4HF0YZYignM4TelFUJSSqQlMLbVE3eWueZKxBilKKOd69c0k5ZbQqnA0na1N8ddwmyYwDIHbH1Ewxp5a5Ux
lksi4H/ctZwaT3EHmkAmOKFKMmt0SJK0GlfVGmat7NQU9C80w8aqCp6m0oSdpQiIGWNEUsuNYUrAp70Z7iDfnMObObaWUsE1dZew9TmsBVcgcJZxy8TeFHeOte2CMkmMK27vDqWoFEJrqpiSYm+KO4wE669BiKwm
gnWu84MpaQu/BgUghCF0IsVdC244M7DdDe2WXzC3jQjsJC0nctxPH8g+0UBPmOD+fD242mrlXiUynALVpeBe4dagXVQf5cCqfkPPoYZHXuhl6Oqro/oAhQdf2935/aQh3esS1JBa0pmPJuCM9gMYt0K/FVefvQrC
4KCvu+5U7Ea0Gts1Vv/0m3V6T1T9xDutZTrd7t3sH3vzXesiuoBzXQAaNBaNo1+CFkF+KV/b8hwnn7eNRQcWyLeqJy5I08ei0V5161AXdy+P4Sevi3s39QcFJbzVxFmR75ruoM4BBp9jlxfrNM+Pzu6uwqB69GIb
GRII0lO76M3ubMGqBqClsofQnlBncUiLfC+45W7p7tKvj6C/jmzJt35CTwUWtTx3F1SvkyKrAIivycOBdRWfPHyQj7LcHk0y1TEDdufFdrO+Qe3X3snQTMI1nXUVv/tYfgi+4KV5bNNit133mT4i3odMhcYqgDk7
IcU9Gb/TaNS2aiSAfEnu77ebZHHrKZN9XTEQi/IbkgKZIRgycc2qOiWc1YJg5Z0xzWmJ58d2c3/fC5+b4xkuUd6Xu7Ibcw38NzdF3KXr3o2x5qAEC9Volp4C7NBJe3QO4K6UxeLtaem9nRihaXIQhgn72D8pmqM7
RA+AY+A+7rZew4Lrtsfiabm9IbFsr65gTr3VHZY9ACS8pWl3XHfOIK7hDdoezYFB4RZsMMgBxH2qMKy75Ju3mHXQIgyX451GqwyGdUjDDmzOG/x0YIq/Z014Y45HYDzivIhNAzab2DRkNlfXTJVJjMUkGg3jvWlo
PQXq/k8TMYky1csZY6jnj7/MU/ttjuhQ31Pu9z2tkrbDXG1TPVGndXuJ4Lx+kNavpb2MbC/bu07pfvUOSq/rk3w3bZXkc/ZhfQX/6wUB4NOTssVRnUZn6jjOr4Grmuzhc8b+7exuvLTHJwGcjVv+glokvG/5qCRr
/oTpvjNAFs4uA4M+Mg3mnACT872ZlD+6qmYuONDZrQF/5AY5M3jr7NZHnOaZzQKuO3cPjMpHHuNnh0rO2TWjx/cZflVcdda81KeCC3MHiOdbsTNUQ5zfg9F9Sr7ey2HaXSXeUNz79Njc5zMBzOfseDLydHUvL4n5
v7JZNs7nkeRmsW8LzyRmcfjG3f/pme+ID27/qYvOXIsROGpLiHM52IT6L+/23dA5cM+76+FdOm5eqr5Len5T3WTU3mKEvdfBAb96iG6An3XT6rYXNRDXu6Nz9Mp4SkAOw+xZq+1UQ2klRGyoFcRiYiX8+Mfv/wc9
YK14
"""
SYS = {}
def add_system(name, mod, E_ref, E_x, labels, usecase, source):
    """Register a molecule: reference (Skala) grid, scout (xTB) grid, and the anchors of the three starting designs."""
    bs = mod.bottleneck_barrier(E_ref, theta); bx = mod.bottleneck_barrier(E_x, theta)
    a = tuple(int(v) for v in bx["hcn"]); b = tuple(int(v) for v in bx["hnc"])            # the two wells according to the scout
    fa = tuple(int(v) for v in bs["hcn"]); fb = tuple(int(v) for v in bs["hnc"])          # the two wells according to Skala
    anchors = dict(scout=[a, b], random=[], edge=[(0, a[1]), (len(theta) - 1, b[1])])
    ka = (E_ref - E_ref[fa]) * mod.HARTREE_KCAL; kb = (E_ref - E_ref[fb]) * mod.HARTREE_KCAL
    SYS[name] = dict(mod=mod, E_REF=E_ref, E_xtb=E_x, REF=float(bs["barrier_kcal"]), bs=bs, bx=bx, a=a, b=b, floor=fa, labels=labels, usecase=usecase, source=source, anchors=anchors,
                     above=dict(scout=float(ka[a]), edge=float(ka[anchors["edge"][0]])), above_other=dict(scout=float(kb[b]), edge=float(kb[anchors["edge"][1]])))
    SYS[name]["edge_misses"] = SYS[name]["above"]["edge"] > 1.0; SYS[name]["scout_hits"] = SYS[name]["above"]["scout"] <= 1.0
    return SYS[name]
_g = np.load(io.BytesIO(base64.b64decode("".join(_GRID_B64.split()))))
assert _g["E_skala"].shape == (N_THETA, N_R) and np.allclose(_g["theta"], theta) and np.allclose(_g["r"], r), "embedded grid does not match the (theta, r) grid of cell 4"
STORED_VINYL_EDGE = json.loads(zlib.decompress(base64.b64decode("".join(_RUNS_B64.split()))))
if RUN_VINYL:
    S_ = add_system("vinyl", fpes_va, _g["E_skala"].copy(), _g["E_xtb"].copy(), ("enol", "keto"), USECASE["vinyl"], "Skala-1.1/def2-svp grid of NB10 (L4, 2026-11-12), embedded")
    assert abs(S_["REF"] - 74.87) < 0.01, S_["REF"]
    for x in STORED_VINYL_EDGE:      # the stored runs were scored against this same reference
        assert abs((x["hist"][-1]["barrier"] - x["hist"][-1]["error"]) - S_["REF"]) < 1e-6
    print(f"vinyl alcohol: Skala barrier {S_['REF']:.2f} kcal/mol on the rigid-recipe grid; {len(STORED_VINYL_EDGE)} stored agent runs (model {sorted({x['model'] for x in STORED_VINYL_EDGE})}), "
          f"median {np.median([len(x['idx']) for x in STORED_VINYL_EDGE]):.0f} Skala calls, largest error {max(abs(x['hist'][-1]['error']) for x in STORED_VINYL_EDGE):.2f} kcal/mol")


## Part A: the new molecule, formamide ⇌ formimidic acid (cells 6–8; a GPU for 7 and 8)
Skipped automatically when `RUN_FORMAMIDE` is off. Cell 6 is the free scout (seconds). Cells 7–8 compute the Skala energy at all 247 grid points; every point is cached on Drive, so a rerun resumes.

In [ ]:
#@title 6. Free scout for formamide: xTB over the whole grid (CPU, seconds), and a check against the design scan
if RUN_FORMAMIDE:
    res_x = fpes.compute_surface(fpes.XTB(), theta, r, f"{WORKDIR}/ref_xtb_formamide_tautomer_{N_THETA}x{N_R}.npz", log_every=1000)
    E_xtb_fi = fpes.fill_nan_grid(res_x["E"], theta, r)
    bx = fpes.bottleneck_barrier(E_xtb_fi, theta)
    print(f"xTB barrier {bx['barrier_kcal']:.2f} kcal/mol (from the amide well, rigid-recipe grid), saddle theta {theta[bx['saddle'][0]]:.0f}, r {r[bx['saddle'][1]]:.2f}")
    print(f"xTB amide well theta {theta[bx['hcn'][0]]:.0f}, r {r[bx['hcn'][1]]:.2f}; iminol well theta {theta[bx['hnc'][0]]:.0f}, r {r[bx['hnc'][1]]:.2f}; E(iminol) - E(amide) = {bx['isomerization_kcal']:.1f} kcal/mol; NaN points: {int(np.isnan(res_x['E']).sum())}")
    print("expected from the design scan: barrier 51.9, saddle theta 80 / r 1.0, amide well theta 20 / r 1.5, iminol well theta 160 / r 1.2, iminol 10.3 above, 0 NaN points")
else:
    print("RUN_FORMAMIDE is off: skipped")


In [ ]:
#@title 7. Skala on the GPU: smoke test and timing for formamide (stop here if this fails). PART A
if RUN_FORMAMIDE:
    if SKALA_STAND_IN:
        sk = fpes.DFT("pbe", "sto-3g", gpu=False); where = "STAND-IN (PBE/STO-3G CPU)"; print("*** DRY RUN: PBE/STO-3G stands in for Skala. Do not report these numbers. ***")
    else:
        try:
            sk = fpes.Skala(BASIS, gpu=bool(GPU)); where = "GPU" if GPU else "CPU"
        except Exception as e:
            print("GPU Skala unavailable, falling back to CPU:", repr(e)[:160]); sk = fpes.Skala(BASIS, gpu=False); where = "CPU"
    i0, j0 = int(np.argmin(np.abs(theta - 20))), int(np.argmin(np.abs(r - 1.5)))          # the amide well of the scout
    for k in range(2):
        sk.reset_guess(); out = sk(fpes.hcn_geometry(theta[i0], r[j0]))
        print(f"[{where}] test {k+1}: E = {out['E']:.6f} Eh, converged {out['converged']}, {out['niter']} iterations, {out['seconds']:.1f} s")
    assert np.isfinite(out["E"]), "Skala returned NaN: check the install cell output and the GPU runtime"
    isad = int(np.argmin(np.abs(theta - 90))); jsad = int(np.argmin(np.abs(r - 1.0)))     # a harder probe near the pass, where SCF trouble would show first
    sk.reset_guess(); o2 = sk(fpes.hcn_geometry(theta[isad], r[jsad]))
    print(f"[{where}] pass-region probe (theta {theta[isad]:.0f}, r {r[jsad]:.2f}): E = {o2['E']:.6f} Eh, converged {o2['converged']}, {o2['niter']} iterations, {o2['seconds']:.1f} s")
    print(f"projected whole-grid time at the slower of the two probes: {N_THETA*N_R*max(out['seconds'], o2['seconds'])/60:.0f} min (Part B uses stored values, so this is all the Skala time of the notebook)")
else:
    print("RUN_FORMAMIDE is off: skipped")


In [ ]:
#@title 8. Skala-1.1 over the whole formamide grid: the oracle. Time-boxed and resumable (rerun this cell to continue). PART A
SK_PATH = f"{WORKDIR}/ref_{'standin' if SKALA_STAND_IN else 'skala'}_formamide_tautomer_{BASIS}_{N_THETA}x{N_R}.npz"
if RUN_FORMAMIDE:
    S_sk = fpes.compute_surface(sk, theta, r, SK_PATH, every=1, log_every=10, max_minutes=SKALA_MAX_MIN)
    _done = int((~np.isnan(S_sk["E"])).sum())
    print(f"\nSkala points done: {_done} of {N_THETA*N_R}" + ("" if _done == N_THETA*N_R else "   <- not complete: rerun this cell until it is (it resumes), then continue"))
else:
    print("RUN_FORMAMIDE is off: skipped")


In [ ]:
#@title 9. The reference surfaces, the three starting designs, and the check that the test means something. Part B starts here (CPU is enough).
FILLED_POINTS = []
if RUN_FORMAMIDE:
    SK_PATH = f"{WORKDIR}/ref_{'standin' if SKALA_STAND_IN else 'skala'}_formamide_tautomer_{BASIS}_{N_THETA}x{N_R}.npz"
    assert os.path.exists(SK_PATH), "no Skala grid for formamide yet: run cells 7-8 on a GPU runtime first"
    assert "E_xtb_fi" in globals(), "run cell 6 (the xTB scout) first"
    _d = dict(np.load(SK_PATH, allow_pickle=True))
    assert _d["E"].shape == (N_THETA, N_R) and np.allclose(_d["theta"], theta) and np.allclose(_d["r"], r), "the saved grid does not match the (theta, r) grid of cell 4"
    _miss = np.argwhere(np.isnan(_d["E"]))
    assert len(_miss) <= 12, f"{len(_miss)} Skala points are still missing: rerun cell 8 on a GPU runtime (up to 12 failed points are tolerated and filled by interpolation)"
    E_REF_FI = _d["E"].copy()
    if len(_miss):
        E_REF_FI = fpes.fill_nan_grid(E_REF_FI, theta, r)
        print(f"WARNING: {len(_miss)} Skala points failed and are filled by interpolation: " + ", ".join(f"(theta {theta[i]:.0f}, r {r[j]:.2f})" for i, j in _miss))
        print("         check that none of them sits near the pass or in a well; if one does, the barrier is an interpolation there.")
    FILLED_POINTS = [(float(theta[i]), float(r[j])) for i, j in _miss]
    _conv = float(np.mean(_d["converged"][~np.isnan(_d["E"])])); _sec = float(np.nanmean(_d["seconds"]))
    print(f"formamide grid: {E_REF_FI.size} points, SCF converged in {100*_conv:.1f}% of them, {_sec:.1f} s per point on average (whole grid {np.nansum(_d['seconds'])/60:.0f} min)")
    if _conv < 1.0: print("WARNING: some SCF cycles did not converge; the grid keeps their last energies. Check the 'converged' array before quoting numbers.")
    add_system("formamide", fpes_fi, E_REF_FI, E_xtb_fi, ("amide", "iminol"), USECASE["formamide"],
               ("STAND-IN PBE/STO-3G grid (dry run)" if SKALA_STAND_IN else f"Skala-1.1/{BASIS} grid from this notebook"))
def _p(ij): return f"(theta {theta[ij[0]]:.0f}, r {r[ij[1]]:.2f})"
print()
for name in MOLS:
    S_ = SYS[name]; A, B = S_["labels"]; bs, bx = S_["bs"], S_["bx"]
    print(f"{name.upper()}  [{S_['source']}]")
    print(f"  reference barrier {S_['REF']:.2f} kcal/mol from the {A} well (scout: {bx['barrier_kcal']:.2f}); pass at {_p(bs['saddle'])} (scout: {_p(bx['saddle'])}); E({B}) - E({A}) = {bs['isomerization_kcal']:.1f} kcal/mol")
    print(f"  {A} well: reference floor at {_p(S_['floor'])}; scout well {_p(S_['a'])} is {S_['above']['scout']:.2f} kcal/mol above that floor; edge anchor {_p(S_['anchors']['edge'][0])} is {S_['above']['edge']:.2f} above it")
    print(f"  {B} well: scout well {_p(S_['b'])} is {S_['above_other']['scout']:.2f} above its floor; edge anchor {_p(S_['anchors']['edge'][1])} is {S_['above_other']['edge']:.2f} above it")
    print("  -> scout start: " + ("INFORMED (the scout well is the reference well to within 1 kcal/mol)" if S_["scout_hits"] else "NOT informed on this molecule: the scout well is more than 1 kcal/mol above the floor, so P1 and P5 are a harder test here; say so"))
    print("  -> edge start:  " + ("MISINFORMED (the anchor misses the reference well by more than 1 kcal/mol): P2 and P4 apply to it" if S_["edge_misses"] else "happens to hit the reference well (within 1 kcal/mol): it counts as an informed start, P2 and P4 use the random start only"))
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, len(MOLS), figsize=(5.6 * len(MOLS), 4.4), constrained_layout=True, squeeze=False)
for ax, name in zip(axs[0], MOLS):
    S_ = SYS[name]; mod = S_["mod"]
    cs = mod.plot_surface(ax, S_["E_REF"], theta, r, title=f"{name}: barrier {S_['REF']:.1f} kcal/mol from the {S_['labels'][0]} well")
    ax.plot(theta[S_["bs"]["saddle"][0]], r[S_["bs"]["saddle"][1]], "r*", ms=13, label="pass")
    for (i, j) in S_["anchors"]["scout"]: ax.plot(theta[i], r[j], "o", mfc="white", mec="k", ms=9)
    for (i, j) in S_["anchors"]["edge"]: ax.plot(theta[i], r[j], "s", mfc="none", mec="white", mew=2, ms=9, clip_on=False)
    ax.plot([], [], "o", mfc="white", mec="k", label="scout-well start points"); ax.plot([], [], "s", mfc="none", mec="grey", mew=2, label="edge-anchor start points")
    ax.legend(fontsize=7, loc="upper center"); fig.colorbar(cs, ax=ax, label="E - E_min (kcal/mol)")
fig.savefig(f"{WORKDIR}/figures/fig_nb12_surfaces_and_starts{TAG_S}.png", dpi=200); plt.show()


## Part B: three kinds of start, every arm (no Skala calls from here; a CPU runtime is enough)
Every 'Skala call' below is a lookup in a stored grid. Starts 0 to SEEDS−1 use the same random numbers in every arm, so all comparisons are paired; the two random points of the scout and edge starts are the first two of the four random points of the random start.

In [ ]:
#@title 10. The three starting designs, the reordered twin, the control, and the scoring functions
BEST_FIXED = "saddle_exploit+delta"        # the classical rule named in advance (best on HCN, CH3NC and vinyl alcohol)
POLICIES = ["random", "uniform", "uncertainty", "path_uncertainty", "saddle_exploit"]
ARMS_CLASSICAL = [p + d for p in POLICIES for d in ("", "+delta")]
FIXED_ARMS = ARMS_CLASSICAL + ["twin", "twin_wells_first", "control"]

def start_design(name, cond):
    """The function that draws the 4 starting points: the condition's fixed anchors, then random points. 'edge' on vinyl alcohol reproduces NB10 exactly."""
    fixed = [tuple(p) for p in SYS[name]["anchors"][cond]]
    def init(theta_, r_, n_init, rng):
        idx = list(fixed)
        while len(idx) < n_init:
            c = (int(rng.integers(len(theta_))), int(rng.integers(len(r_))))
            if c not in idx: idx.append(c)
        return idx
    return init

def _install_wells_first(mod):
    """Reordered twin: the prompt's decision order with step 3 (unsampled basin minimum) moved ahead of step 2 (on-path candidate). One extra rule, nothing else."""
    if getattr(mod, "_wells_first", False): return
    orig = mod.frozen_menu_choice
    def choice(J, menu, rule="order"):
        if rule == "wells_first":
            sad = next((m for m in menu if m["tag"] == "saddle guess"), None)
            if sad is not None and sad["sigma"] > 0.5: return dict(name="pick", input=dict(id=sad["id"], why="rule 1: saddle guess unsampled"))
            basin = next((m for m in menu if "basin" in m["tag"]), None)
            if basin is not None: return dict(name="pick", input=dict(id=basin["id"], why="rule 3 (moved up): basin minimum unsampled"))
            rule = "order"
        return orig(J, menu, rule)
    mod.frozen_menu_choice = choice; mod._wells_first = True
for mod_ in MODS.values(): _install_wells_first(mod_)

def new_judge(name, cond, s):
    S_ = SYS[name]; mod = S_["mod"]; mod.initial_design = start_design(name, cond)
    return mod.JudgeRun(mod.GridOracle(S_["E_REF"], theta, r), theta, r, S_["E_xtb"], BUDGET, ref_barrier=S_["REF"], baseline=S_["E_xtb"], seed=s)

def run_control(name, cond, s, n_init=4):
    """Anchor-first control (as in NB11): the same 4 starting points, then Skala calls at the scout's saddle and its two wells, then the best classical rule."""
    S_ = SYS[name]; mod = S_["mod"]; mod.initial_design = start_design(name, cond); E, X, REF = S_["E_REF"], S_["E_xtb"], S_["REF"]
    rng = np.random.default_rng(s); orc = mod.GridOracle(E, theta, r); idx = mod.initial_design(theta, r, n_init, rng); y = [orc(i, j) for i, j in idx]
    forced = [tuple(int(v) for v in S_["bx"][k_]) for k_ in ("saddle", "hcn", "hnc")]; hist = []
    while True:
        mean, std = mod.gp_predict(theta, r, idx, y, baseline=X)
        for p_, (i, j) in enumerate(idx): mean[i, j] = y[p_]
        est = mod.bottleneck_barrier(mean, theta)["barrier_kcal"]; hist.append(dict(calls=len(idx), barrier=est, error=est - REF))
        if len(idx) >= BUDGET: break
        nxt = None
        while forced and nxt is None:
            c_ = forced.pop(0)
            if c_ not in idx: nxt = c_
        if nxt is None: nxt = tuple(int(v) for v in mod.choose_next("saddle_exploit", theta, r, idx, mean, std, rng=rng))
        idx.append(nxt); y.append(orc(*nxt))
    return hist, idx

def run_fixed(name, cond, arm, s):
    S_ = SYS[name]; mod = S_["mod"]; E, X, REF = S_["E_REF"], S_["E_xtb"], S_["REF"]
    if arm in ARMS_CLASSICAL:
        mod.initial_design = start_design(name, cond)
        hist, idx_, _ = mod.run_policy(arm.replace("+delta", ""), mod.GridOracle(E, theta, r), theta, r, BUDGET, seed=s, baseline=X if arm.endswith("+delta") else None, ref_barrier=REF)
    elif arm in ("twin", "twin_wells_first"):
        J = new_judge(name, cond, s); mod.run_menu_agent(J, S_["usecase"], chat_factory=None, frozen="order" if arm == "twin" else "wells_first", verbose=False)
        hist, idx_ = J.hist, J.idx
    else:
        hist, idx_ = run_control(name, cond, s)
    return dict(seed=s, hist=hist, idx=[[int(i), int(j)] for i, j in idx_], ref=REF)

def calls_to_tol(hist, tol=1.0):
    errs = [abs(h["error"]) if h["error"] is not None and np.isfinite(h["error"]) else np.inf for h in hist]
    for q in range(len(errs)):
        if all(e <= tol for e in errs[q:]): return hist[q]["calls"]
    return None
def stop_index(hist, tol=STOP_TOL, window=STOP_WINDOW, min_calls=STOP_MIN_CALLS):
    b = [h["barrier"] for h in hist]
    for q in range(len(hist)):
        if hist[q]["calls"] < min_calls or q < window: continue
        w = b[q - window:q + 1]
        if np.all(np.isfinite(w)) and (max(w) - min(w)) < tol: return q
    return len(hist) - 1
def check_ref(store, what):
    """Saved runs must have been scored against the reference barrier that is loaded now."""
    for key, runs_ in store.items():
        name = key.split("|")[0]
        if name in SYS:
            bad = [x["seed"] for x in runs_ if abs(x.get("ref", SYS[name]["REF"]) - SYS[name]["REF"]) > 1e-6]
            assert not bad, f"{what}: runs under '{key}' were made against a different reference barrier (a different Skala grid). Move the file away to start fresh."
# the edge start on vinyl alcohol must be the NB10 start, point for point
if "vinyl" in SYS:
    for x in STORED_VINYL_EDGE:
        assert [tuple(p) for p in x["idx"][:4]] == start_design("vinyl", "edge")(theta, r, 4, np.random.default_rng(x["seed"])), "edge start differs from NB10"
    print("edge start on vinyl alcohol reproduces the NB10 starting points for all stored runs")
for name in MOLS:
    print(name, {c: [f"({theta[i]:.0f}, {r[j]:.2f})" for i, j in SYS[name]["anchors"][c]] or "none (4 random points)" for c in CONDITIONS})


In [ ]:
#@title 11. The fixed arms under every start: ten classical rules, frozen twin, reordered twin, control (CPU, about 10-20 min, resumable)
FIX_OUT = f"{WORKDIR}/fixed_runs{TAG_S}.json"
FIXED = json.load(open(FIX_OUT)) if os.path.exists(FIX_OUT) else {}
check_ref(FIXED, "fixed_runs")
try:
    from threadpoolctl import threadpool_limits     # one maths thread: these fits are tiny, and several threads only get in each other's way
except Exception:
    import contextlib
    def threadpool_limits(limits=None): return contextlib.nullcontext()
t0 = time.time(); n_new = 0
with threadpool_limits(limits=1):
    for name in MOLS:
        for cond in CONDITIONS:
            for arm in FIXED_ARMS:
                key = f"{name}|{cond}|{arm}"; have = {x["seed"] for x in FIXED.get(key, [])}; todo = [s for s in range(SEEDS) if s not in have]
                for s in todo:
                    FIXED.setdefault(key, []).append(run_fixed(name, cond, arm, s)); n_new += 1
                if todo: json.dump(FIXED, open(FIX_OUT, "w"))
            print(f"{name:10s} {cond:7s} done   ({(time.time()-t0)/60:.1f} min)")
print(f"{n_new} new fixed runs; file {FIX_OUT}")
# the NB10 check: best classical rule, edge start, vinyl alcohol -> errors at stop must reproduce NB10 (1 of 8 within 1 kcal/mol, median 1.69)
if "vinyl" in SYS and SEEDS >= 8:
    _e = [abs(x["hist"][stop_index(x["hist"])]["error"]) for x in sorted(FIXED[f"vinyl|edge|{BEST_FIXED}"], key=lambda x: x["seed"])[:8]]
    print(f"check against NB10 (vinyl, edge start, {BEST_FIXED}): within 1 kcal/mol in {sum(e <= 1 for e in _e)} of 8, median error {np.median(_e):.2f}  (NB10: 1 of 8, 1.69)")
    assert sum(e <= 1 for e in _e) == 1 and abs(np.median(_e) - 1.69) < 0.02, "the classical rule no longer reproduces NB10: library versions changed the surrogate; tell Claude before reading further"


In [ ]:
#@title 12. The LLM agent under every start (Gemini; resumable: every finished run is saved, rerun this cell after a disconnect)
import concurrent.futures as cf
AG_OUT = f"{WORKDIR}/agent_runs{TAG}.json"
agent_runs = json.load(open(AG_OUT)) if os.path.exists(AG_OUT) else {}
if "vinyl" in SYS and not RERUN_VINYL_EDGE and "vinyl|edge" not in agent_runs:
    agent_runs["vinyl|edge"] = [dict(x, ref=SYS["vinyl"]["REF"], source="NB10 stored run (L4 grid, 2026-11-12)") for x in STORED_VINYL_EDGE]
    print("vinyl / edge start: using the 8 stored NB10 agent runs (set RERUN_VINYL_EDGE to run them again)")
if RERUN_VINYL_EDGE and "vinyl|edge" in agent_runs:      # keep only runs made by this notebook
    agent_runs["vinyl|edge"] = [x for x in agent_runs["vinyl|edge"] if not str(x.get("source", "")).startswith("NB10 stored")]
check_ref(agent_runs, "agent_runs")
def done(name, cond, s): return any(x["seed"] == s for x in agent_runs.get(f"{name}|{cond}", []))
plan = [(s, name, cond) for s in range(SEEDS) for name in MOLS for cond in CONDITIONS if not done(name, cond, s)]     # start-major: an interrupted session leaves balanced cells
print(f"{len(plan)} agent runs to do" + (f", about {len(plan)*4/max(1, PARALLEL):.0f} min at 4 min per run and {PARALLEL} in parallel" if plan and not MOCK else ""))
jobs = [(s, name, cond, new_judge(name, cond, s)) for s, name, cond in plan]      # the starting points are drawn here, in the main thread
def one(job):
    s, name, cond, J = job; S_ = SYS[name]; t0 = time.time()
    try:
        S_["mod"].run_menu_agent(J, S_["usecase"], chat_factory=make_chat[name], scaffold=True, verbose=False)
    except Exception as ex:
        return job, repr(ex)[:160], time.time() - t0
    return job, None, time.time() - t0
t_all = time.time(); n_new = 0
with cf.ThreadPoolExecutor(max_workers=max(1, int(PARALLEL))) as pool:
    for fut in cf.as_completed([pool.submit(one, j) for j in jobs]):
        (s, name, cond, J), err, secs = fut.result(); mod = SYS[name]["mod"]
        if err:
            print(f"  {name} {cond} start {s} FAILED ({err}); not saved, rerun the cell to retry"); continue
        agent_runs.setdefault(f"{name}|{cond}", []).append(dict(seed=s, model=J.model, hist=J.hist, decisions=J.decisions, idx=[list(map(int, p)) for p in J.idx], notes=J.notes,
            menu_log=J.menu_log, summary=J.summary, tokens=dict(J.tokens), cost_usd=mod.cost_usd(J.model, J.tokens), seconds=secs, ref=SYS[name]["REF"], source="NB12"))
        json.dump(agent_runs, open(AG_OUT, "w"), default=str); n_new += 1
        print(f"{name:10s} {cond:7s} start {s}: barrier error {J.hist[-1]['error']:+.2f} after {len(J.idx):2d} calls | {J.model} | {secs:.0f}s")
print(f"\n{n_new} new runs in {(time.time()-t_all)/60:.1f} min; {sum(len(v) for v in agent_runs.values())} runs saved in {AG_OUT}")
_models = sorted({str(x["model"]) for v in agent_runs.values() for x in v})
print("models that answered:", _models)
if not MOCK and any(m != (MODEL or MODEL_FAST) for m in _models):
    _fb = [(k, x["seed"], x["model"]) for k, v in agent_runs.items() for x in v if x["model"] != (MODEL or MODEL_FAST)]
    print("NOTE: some runs ended on a fallback model (rate limit or outage):", _fb, "\n      for a clean one-model comparison delete those entries from the json and rerun this cell with PARALLEL = 1.")


In [ ]:
#@title 13. Per-start scores for every arm and every start: error and calls at stop, calls to 1 kcal/mol, and the call that first measured the reference well
import pandas as pd, math
from scipy import stats
BIG = BUDGET + 1
def well_call(run, name):
    """Number of the Skala call that first measured the reference well: a point in the reference basin (theta < 45) within WELL_TOL kcal/mol of the basin floor. BIG = never."""
    S_ = SYS[name]; E = S_["E_REF"]; floor = E[S_["floor"]]
    for k, (i, j) in enumerate(run["idx"], 1):
        if theta[i] < 45.0 and (E[i, j] - floor) * fpes.HARTREE_KCAL <= WELL_TOL: return k
    return BIG
def row_of(run, name, self_stopping):
    h = run["hist"]; q = len(h) - 1 if self_stopping else stop_index(h); k = h[q]["calls"]; to1 = calls_to_tol(h, 1.0); wc = well_call(run, name)
    return dict(seed=run["seed"], stop_calls=k, stop_err=abs(h[q]["error"]), within1=bool(abs(h[q]["error"]) <= 1.0), to1=BIG if to1 is None else to1,
                well_call=wc, well_by_stop=bool(wc <= k), calls_end=len(run["idx"]))
T = {}      # (molecule, start, arm) -> DataFrame by start number
def _df(runs_, name, self_stopping):
    runs_ = [x for x in runs_ if x["seed"] < SEEDS]
    return pd.DataFrame([row_of(x, name, self_stopping) for x in runs_]).set_index("seed").sort_index() if runs_ else None
for key, runs_ in FIXED.items():
    name, cond, arm = key.split("|")
    if name in SYS and (d_ := _df(runs_, name, False)) is not None: T[(name, cond, arm)] = d_
for key, runs_ in agent_runs.items():
    name, cond = key.split("|")
    if name in SYS and (d_ := _df(runs_, name, True)) is not None: T[(name, cond, "agent")] = d_
# pairing check: within a molecule and a kind of start, every arm began from the same four points
for name in MOLS:
    for cond in CONDITIONS:
        first = {}
        for arm in FIXED_ARMS + ["agent"]:
            src = agent_runs.get(f"{name}|{cond}", []) if arm == "agent" else FIXED.get(f"{name}|{cond}|{arm}", [])
            for x in src: first.setdefault(x["seed"], set()).add(tuple(map(tuple, x["idx"][:4])))
        assert all(len(v) == 1 for v in first.values()), f"{name} {cond}: arms did not share their starting points"
ARM_SHOW = [("agent", "LLM agent (menu)" if not MOCK else "MOCK agent (= frozen twin)"), (BEST_FIXED, "best classical rule (named in advance)"), ("twin", "frozen twin (prompt order, literal)"),
            ("twin_wells_first", "reordered twin (well before path)"), ("control", "anchor-first control")]
def best_posthoc(name, cond):
    c = [(a, T[(name, cond, a)]) for a in ARMS_CLASSICAL if (name, cond, a) in T]
    return max(c, key=lambda t: (t[1].within1.sum(), -t[1].stop_err.median()))[0] if c else None
rows = []
for name in MOLS:
    for cond in CONDITIONS:
        arms = [(a, lab) for a, lab in ARM_SHOW] + ([(best_posthoc(name, cond), "best of the ten classical rules (picked afterwards)")] if best_posthoc(name, cond) else [])
        for arm, lab in arms:
            df = T.get((name, cond, arm))
            if df is None: continue
            rows.append(dict(molecule=name, start=cond, arm=lab + (f" = {arm}" if "afterwards" in lab else ""), n=len(df), within_1=f"{int(df.within1.sum())}/{len(df)}",
                err_median=round(df.stop_err.median(), 2), err_max=round(df.stop_err.max(), 2), calls_at_stop=f"{df.stop_calls.median():.0f} [{df.stop_calls.quantile(.25):.0f}-{df.stop_calls.quantile(.75):.0f}]",
                calls_to_1=f"{df.to1.median():.0f}", well_measured_by_stop=f"{int(df.well_by_stop.sum())}/{len(df)}", well_call_median=f"{df.well_call.median():.0f}"))
TAB = pd.DataFrame(rows); pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 70)
if MOCK: print("*** MOCK mode: the agent rows are the frozen twin with its own stop. Pipeline check only. ***\n")
print(f"within_1 = starts within 1 kcal/mol when the search stops. calls include the 4 starting points; {BIG} = never. Agents stop themselves; all other arms use the stopping rule.\n")
for name in MOLS:
    for cond in CONDITIONS:
        print(f"--- {name}, {COND_LABEL[cond]}" + ("" if cond != "edge" else ("  [anchor misses the reference well]" if SYS[name]["edge_misses"] else "  [anchor happens to hit the reference well]")))
        print(TAB[(TAB.molecule == name) & (TAB.start == cond)].drop(columns=["molecule", "start"]).to_string(index=False), "\n")
# the compact picture: within 1 kcal/mol, arm x start
for name in MOLS:
    piv = pd.DataFrame({COND_LABEL[c]: {lab: (f"{int(T[(name, c, a)].within1.sum())}/{len(T[(name, c, a)])}" if (name, c, a) in T else "-") for a, lab in ARM_SHOW} for c in CONDITIONS})
    print(f"=== {name}: starts within 1 kcal/mol at stop ===\n{piv.to_string()}\n")


In [ ]:
#@title 14. Paired tests and the prediction scorecard
def pair(name, cond, a, b, col):
    x, y = T.get((name, cond, a)), T.get((name, cond, b))
    if x is None or y is None: return None
    sd = x.index.intersection(y.index); d = (x.loc[sd, col] - y.loc[sd, col]).values.astype(float)
    if len(d) < 2: return None
    p = 1.0 if np.all(d == 0) else stats.wilcoxon(d, zero_method="zsplit").pvalue
    return dict(n=len(d), median_diff=round(float(np.median(d)), 3), a_better=int((d < 0).sum()), tie=int((d == 0).sum()), a_worse=int((d > 0).sum()), wilcoxon_p=round(float(p), 3))
METRICS = (("stop_err", "error at stop"), ("stop_calls", "calls at stop"), ("to1", "calls to 1 kcal/mol"), ("well_call", "call that first measured the reference well"))
cmp_rows = []
for name in MOLS:
    for cond in CONDITIONS:
        for b in (BEST_FIXED, "twin", "twin_wells_first", "control"):
            for col, lab in METRICS:
                r_ = pair(name, cond, "agent", b, col)
                if r_: cmp_rows.append(dict(molecule=name, start=cond, a="agent", b=b, metric=lab, **r_))
CMP = pd.DataFrame(cmp_rows)
print("difference = agent minus the other arm; NEGATIVE = agent better. With n starts the smallest attainable p is 2/2^n (n = 8: 0.008).\n")
if len(CMP):
    for name in MOLS:
        print(f"--- {name}: agent against the best classical rule and against its frozen twin")
        print(CMP[(CMP.molecule == name) & CMP.b.isin([BEST_FIXED, "twin"])].drop(columns=["molecule", "a"]).to_string(index=False), "\n")
        print(f"--- {name}: agent against the reordered twin and the control (no prediction; how much does a one-line change of the fixed order already buy?)")
        print(CMP[(CMP.molecule == name) & CMP.b.isin(["twin_wells_first", "control"]) & CMP.metric.isin(["error at stop", "calls at stop"])].drop(columns=["molecule", "a"]).to_string(index=False), "\n")

# ---- the scorecard: P1-P5 of the introduction, evaluated per molecule. Thresholds scale with the number of starts (8 starts: 6, 3 and 7).
OKF = lambda n: math.ceil(0.75 * n); LO = lambda n: math.floor(0.4 * n); HI = lambda n: math.ceil(0.85 * n)
def cnt(name, cond, arm, col="within1"):
    df = T.get((name, cond, arm)); return (None, 0) if df is None or len(df) < min(SEEDS, 8) else (int(df[col].sum()), len(df))
SCORE = []
def add(name, pid, statement, observed, ok, known):
    SCORE.append(dict(molecule=name, prediction=pid, statement=statement, observed=observed, verdict="incomplete" if ok is None else ("HOLDS" if ok else "FAILS"), known_before_this_notebook=known))
for name in MOLS:
    S_ = SYS[name]; unin = ["random"] + (["edge"] if S_["edge_misses"] else []); vin = name == "vinyl"
    k, n = cnt(name, "scout", BEST_FIXED)
    add(name, "P1", "informed start: best classical rule within 1 kcal/mol in at least 6 of 8", f"{k}/{n}", None if k is None else k >= OKF(n), "yes (pilot)" if vin else "no")
    obs, ok = [], True
    for cond in unin:
        for arm in (BEST_FIXED, "twin"):
            k, n = cnt(name, cond, arm)
            if k is None: ok = None; continue
            obs.append(f"{cond}: {'classical' if arm == BEST_FIXED else 'twin'} {k}/{n}"); ok = ok if ok is None else (ok and k <= LO(n))
    add(name, "P2", "no reference well in the start (" + " and ".join(unin) + "): best classical rule and frozen twin each within 1 kcal/mol in at most 3 of 8", "; ".join(obs), ok, "yes (pilot)" if vin else "no")
    obs, ok = [], True
    for cond in CONDITIONS:
        k, n = cnt(name, cond, "agent")
        if k is None: ok = None; continue
        obs.append(f"{cond} {k}/{n}"); ok = ok if ok is None else (ok and k >= HI(n))
    add(name, "P3", "agent within 1 kcal/mol in at least 7 of 8 under every start", "; ".join(obs), ok, "edge only (NB10)" if vin else "no")
    obs, ok = [], True
    for cond in unin:
        k, n = cnt(name, cond, "agent", "well_by_stop"); r_ = pair(name, cond, "agent", "twin", "well_call")
        if k is None or r_ is None: ok = None; continue
        ma, mt = T[(name, cond, "agent")].well_call.median(), T[(name, cond, "twin")].well_call.median()
        obs.append(f"{cond}: well measured by stop {k}/{n}; first at call {ma:.0f} (agent) vs {mt:.0f} (twin), p {r_['wilcoxon_p']}"); ok = ok if ok is None else (ok and k >= HI(n) and ma < mt)
    add(name, "P4", "mechanism: under " + " and ".join(unin) + " starts the agent measures the reference well before it stops in at least 7 of 8, and earlier than its twin", "; ".join(obs), ok, "edge only (NB10)" if vin else "no")
    a_, f_ = T.get((name, "scout", "agent")), T.get((name, "scout", BEST_FIXED))
    if a_ is None or f_ is None or len(a_) < min(SEEDS, 8): add(name, "P5", "robustness costs nothing with an informed start", "", None, "no")
    else:
        add(name, "P5", "informed start: agent error not worse than the best classical rule (medians within 0.5 kcal/mol) and no more calls at stop",
            f"error {a_.stop_err.median():.2f} vs {f_.stop_err.median():.2f}; calls {a_.stop_calls.median():.0f} vs {f_.stop_calls.median():.0f}",
            bool(a_.stop_err.median() <= f_.stop_err.median() + 0.5 and a_.stop_calls.median() <= f_.stop_calls.median()), "no")
SC = pd.DataFrame(SCORE)
print("=" * 30, "PREDICTION SCORECARD", "=" * 30)
if MOCK: print("*** MOCK mode: the agent is the frozen twin, so P3-P5 below say nothing about the real agent. ***")
if SKALA_STAND_IN: print("*** stand-in surface for formamide: its rows are a pipeline check only. ***")
for name in MOLS:
    print(f"\n{name}:")
    for _, q in SC[SC.molecule == name].iterrows():
        print(f"  {q.prediction}  {q.verdict:10s} {q.statement}\n      observed: {q.observed}   [known before this notebook: {q.known_before_this_notebook}]")
def _v(pid): return list(SC[SC.prediction == pid].verdict)
print("\nHEADLINE:")
if "incomplete" in _v("P2") + _v("P3"): print("  not all runs are in yet: finish cells 11-12 and rerun cells 13-14.")
elif all(v == "HOLDS" for v in _v("P2") + _v("P3")):
    print("  P2 and P3 hold on every molecule run: fixed rules need the reference well in their starting design, the agent does not." + ("  P5 holds too: the robustness costs nothing when the start is good." if all(v == "HOLDS" for v in _v("P5")) else "  P5 does not hold everywhere: report the price of robustness with an informed start."))
elif all(v == "HOLDS" for v in _v("P3")): print("  The agent is robust everywhere (P3), but a fixed rule also copes without the well on at least one molecule (P2 fails there): claim robustness of the agent, not failure of the fixed rules.")
else: print("  P3 fails somewhere: the agent is not robust to every start. Report the cells where it fails; the NB10 advantage is then specific to the edge start.")


In [ ]:
#@title 15. Figures: who gets within 1 kcal/mol under which start, the errors behind it, and when the reference well was measured
import matplotlib.pyplot as plt
COL = {"agent": "#D97A12", BEST_FIXED: "#555555", "twin": "#E8B77A", "twin_wells_first": "#4C9BD6", "control": "#6B4FD8"}
SHORT = {"agent": "agent" if not MOCK else "MOCK\nagent", BEST_FIXED: "best\nclassical", "twin": "frozen\ntwin", "twin_wells_first": "reordered\ntwin", "control": "control"}
def cond_title(name, c): return {"scout": "scout wells in the start\n(informed)", "random": "four random points\n(uninformed)", "edge": "edge anchors\n(" + ("misinformed" if SYS[name]["edge_misses"] else "happens to be informed") + ")"}[c]
NOTE = ("  [MOCK agent]" if MOCK else "") + ("  [formamide: STAND-IN surface]" if SKALA_STAND_IN else "")
# Figure 2: counts
fig, axs = plt.subplots(1, len(MOLS), figsize=(6.4 * len(MOLS), 4.2), constrained_layout=True, squeeze=False)
for ax, name in zip(axs[0], MOLS):
    w = 0.16
    for k, (arm, _) in enumerate(ARM_SHOW):
        for c_i, cond in enumerate(CONDITIONS):
            df = T.get((name, cond, arm))
            if df is None: continue
            v = int(df.within1.sum()); x = c_i + (k - 2) * w
            ax.bar(x, v, w * 0.92, color=COL[arm], label=SHORT[arm].replace("\n", " ") if c_i == 0 else None); ax.text(x, v + 0.12, str(v), ha="center", fontsize=8)
    ax.set_xticks(range(len(CONDITIONS))); ax.set_xticklabels([cond_title(name, c) for c in CONDITIONS], fontsize=8); ax.set_ylim(0, SEEDS + 1.2)
    ax.set_ylabel(f"starts within 1 kcal/mol at stop (of {SEEDS})"); ax.set_title(name); ax.legend(fontsize=7, ncol=5, loc="upper center")
fig.suptitle("Fixed rules need the reference well in their start; does the agent?" + NOTE, fontsize=10)
fig.savefig(f"{WORKDIR}/figures/fig_nb12_within1{TAG}.png", dpi=200); plt.show()
# Figures 3 and 4: per-start dots
for col, lab, logy, fn in (("stop_err", "|barrier error| at stop (kcal/mol)", True, "error_at_stop"), ("well_call", f"Skala call that first measured the reference well ({BIG} = never)", False, "well_call")):
    fig, axs = plt.subplots(len(MOLS), len(CONDITIONS), figsize=(12, 3.6 * len(MOLS)), constrained_layout=True, squeeze=False); rng_ = np.random.default_rng(0)
    for i, name in enumerate(MOLS):
        for j, cond in enumerate(CONDITIONS):
            ax = axs[i, j]
            for k, (arm, _) in enumerate(ARM_SHOW):
                df = T.get((name, cond, arm))
                if df is None: continue
                v = df[col].values.astype(float)
                ax.plot(k + rng_.uniform(-0.15, 0.15, len(v)), np.maximum(v, 0.01) if logy else v, "o", color=COL[arm], alpha=0.85)
                ax.plot([k - 0.25, k + 0.25], [np.median(v)] * 2, "k-", lw=2)
            ax.set_xticks(range(len(ARM_SHOW))); ax.set_xticklabels([SHORT[a] for a, _ in ARM_SHOW], fontsize=7); ax.set_title(f"{name}: " + cond_title(name, cond).replace("\n", " "), fontsize=9)
            if logy: ax.set_yscale("log"); ax.axhline(1, color="k", ls=":")
            else: ax.set_ylim(0, BIG + 1); ax.axhline(4.5, color="k", ls=":", lw=0.8)
            if j == 0: ax.set_ylabel(lab, fontsize=8)
    fig.suptitle(("Error when the search stops" if logy else "When did the search first measure the well the barrier is counted from? (dotted line: end of the 4 starting points)") + "; one dot per start, black bar = median" + NOTE, fontsize=10)
    fig.savefig(f"{WORKDIR}/figures/fig_nb12_{fn}{TAG}.png", dpi=200); plt.show()


In [ ]:
#@title 16. Summary files and download
import shutil, glob
per_start = []
for (name, cond, arm), df in T.items():
    for sd_, q in df.iterrows(): per_start.append(dict(molecule=name, start=cond, arm=arm, seed=sd_, **{k: (bool(q[k]) if k in ("within1", "well_by_stop") else float(q[k])) for k in df.columns}))
pd.DataFrame(per_start).to_csv(f"{WORKDIR}/nb12_per_start{TAG}.csv", index=False)
summary = dict(notebook="NB12 start-design test", starts=SEEDS, budget=BUDGET, conditions=CONDITIONS, mock_llm=MOCK, skala_stand_in=SKALA_STAND_IN, best_fixed_named_in_advance=BEST_FIXED,
    stop_rule=dict(tol=STOP_TOL, window=STOP_WINDOW, min_calls=STOP_MIN_CALLS), well_tol_kcal=WELL_TOL,
    models=sorted({str(x["model"]) for v in agent_runs.values() for x in v}),
    systems={name: dict(source=S_["source"], reference_barrier_kcal=S_["REF"], scout_barrier_kcal=float(S_["bx"]["barrier_kcal"]), isomerization_kcal=float(S_["bs"]["isomerization_kcal"]),
                        pass_theta_r=[float(theta[S_["bs"]["saddle"][0]]), float(r[S_["bs"]["saddle"][1]])], reference_floor_theta_r=[float(theta[S_["floor"][0]]), float(r[S_["floor"][1]])],
                        start_anchors_theta_r={c: [[float(theta[i]), float(r[j])] for i, j in S_["anchors"][c]] for c in CONDITIONS},
                        anchor_above_reference_floor_kcal=S_["above"], anchor_above_other_floor_kcal=S_["above_other"], edge_misses=bool(S_["edge_misses"]), scout_hits=bool(S_["scout_hits"]),
                        module_sha256=SHA["fpes_va" if name == "vinyl" else "fpes_fi"]) for name, S_ in SYS.items()},
    vinyl_edge_agent_runs="stored NB10 runs" if not RERUN_VINYL_EDGE else "rerun in NB12", filled_skala_points_formamide=FILLED_POINTS,
    table=TAB.to_dict(orient="records"), paired=CMP.to_dict(orient="records"), scorecard=SC.to_dict(orient="records"),
    tokens_in=int(sum(x["tokens"]["input"] for k, v in agent_runs.items() for x in v if x.get("source") == "NB12")), tokens_out=int(sum(x["tokens"]["output"] for k, v in agent_runs.items() for x in v if x.get("source") == "NB12")),
    note="Rigid-recipe grids 19x13; errors are against the reference barrier on the same grid. Predictions P1-P5 were written before any agent run under the scout and random starts and before any Skala energy of formamide; "
         "the fixed-rule results on vinyl alcohol were known from a CPU pilot. The second molecule and its direction were chosen after a pilot on xTB and PBE/B3LYP stand-in surfaces.")
json.dump(summary, open(f"{WORKDIR}/nb12_summary{TAG}.json", "w"), indent=1, default=float)
out = "/content/nb12_results" if os.path.isdir("/content") else os.path.abspath("./nb12_results"); shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
keep = [f"fixed_runs{TAG_S}.json", f"agent_runs{TAG}.json", f"nb12_summary{TAG}.json", f"nb12_per_start{TAG}.csv"] + [os.path.basename(p) for p in glob.glob(f"{WORKDIR}/ref_*formamide_tautomer*.npz")] \
     + [os.path.relpath(p, WORKDIR) for p in glob.glob(f"{WORKDIR}/figures/fig_nb12*.png")]
for f in keep:
    src = f"{WORKDIR}/{f}"
    if os.path.isfile(src):
        os.makedirs(os.path.dirname(f"{out}/{f}") or out, exist_ok=True); shutil.copy(src, f"{out}/{f}"); print("added", f)
shutil.make_archive(out, "zip", out)
shutil.copy(out + ".zip", f"{WORKDIR}/nb12_results{TAG}.zip"); print("copy of the zip on Drive:", f"{WORKDIR}/nb12_results{TAG}.zip")
try:
    from google.colab import files; files.download(out + ".zip")
except Exception: print("zip is at", out + ".zip", "(and on Drive, see above)")


## Reading the result
Cell 14 prints the scorecard. Read it together with the compact table at the end of cell 13.

| what you see | what to write in Paper B2 |
|---|---|
| **P2 and P3 hold on both molecules** (fixed rules at most 3 of 8 without the well, agent at least 7 of 8 everywhere) | The headline: "Fixed acquisition rules reach the barrier only when the reference well is already in their starting design; the language-model agent reaches it from informed, uninformed and misinformed starts, on two reactions." Table 2 of the manuscript gets its agent column, and the starting-design section becomes the main result instead of a caveat. |
| P5 holds as well (with the scout start the agent is as accurate and uses no more calls) | Add: "the robustness has no cost when the start is good". If the agent also stops earlier with the same error, say that, as on HCN and CH₃NC. |
| P3 holds, P2 fails on formamide (a fixed rule copes without the well there) | Claim the agent's robustness on both molecules, and say that on formamide a fixed rule was also robust. Still a consistent, publishable picture; the vinyl contrast stays. |
| P3 fails under the random start | The NB10 advantage was specific to the edge start. Report it as such; the paper becomes a study of how starting designs decide the outcome, with the agent as one of the arms. |
| The reordered twin matches the agent everywhere | The agent's contribution is then fully explained by one reordering of its own default order. The honest claim: "the model found, unprompted, the ordering that a fixed rule needs; hard-coding it afterwards works as well". Pilot on vinyl says the reordered twin gets 5 to 7 of 8, so the agent has room above it. |

**P4 is the mechanism figure.** If the agent measures the reference well within a few calls of the start and the frozen twin does not, Figure "well_call" shows *why* the counts differ, with no interpretation needed.

**Before reading anything:** cell 9 prints, for each molecule, how far each starting anchor is above the floor of the reference well. The prediction P2 is only about starts that miss the well by more than 1 kcal/mol; if the edge anchor happens to hit the well on formamide, the notebook says so and uses the random start alone.

**Caveats to carry into the text.** Two reactions, both 1,3 hydrogen shifts, one model (Gemini 3.8 Flash), 8 starts, one stopping rule with three settings. Rigid-recipe grids: the barriers are grid numbers, not reaction barriers. The fixed-rule results on vinyl alcohol were known before the predictions were written, and the second molecule was chosen after a pilot on cheap surfaces (both stated in the introduction; put them in Methods). The vinyl edge-start agent runs are the stored NB10 runs unless `RERUN_VINYL_EDGE` is on. The agent's prompt names the scout's wells; the fixed rules get the same scout surface numerically but no instruction to go there, and the scout start is what gives them that information.
